# 🧭 24 · Bayesian estimation and maximum a posteriori
### The exponential family and sufficient statistics, Bayes' theorem for parameters, the frequentist and Bayesian schools, pseudocounts, the Beta distribution, conjugate priors (Beta–Bernoulli, Gamma–Poisson, Normal–Normal), maximum a posteriori estimation, posterior means and credible intervals

**BiologyToMath** · Part F — Statistics: from data to estimates · Module 24 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/24_bayesian_estimation_map.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.special`), SymPy, pandas, Matplotlib |
| **Prerequisites** | [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb) (likelihood, MLE recipe), [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) (Bayes' rule), [Module 18 · 🎰 Random variables and their distributions](18_random_variables_distributions.ipynb) (distributions) and [Module 22 · 📏 Sampling: the law of large numbers, the CLT and confidence intervals](22_sampling_lln_clt_confidence.ipynb) (CLT, confidence intervals) |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Recap of the central limit theorem; the distributions met so far as members of the exponential family $P(X\mid\theta) = h(X)e^{\eta(\theta)\cdot T(X) - B(\theta)}$ (binomial, Poisson and normal written in this form), sufficient statistics and why the family is useful; Bayes' theorem and the screening-test example with a known prior; the frequentist versus Bayesian schools (meaning of probability, likelihood only vs prior × likelihood, no universally accepted prior); the Bayesian method; pseudocounts (100 tosses, 0 heads); the Beta distribution; the Beta posterior (conjugacy with the Bernoulli/binomial); maximum a posteriori (MAP) estimation as a point estimate ("partially Bayesian"); the known-prior example (a drawer of three types of coins); MAP with an unknown prior (Bernoulli with a Beta prior); the discrete-prior example; conjugate priors and hyperparameters; MAP for the Poisson with a Gamma prior and for the normal mean with a normal prior |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All data are **SIMULATED** or typed-in worked examples: the drawer of coins, the screening test and the discrete-prior example are classic textbook numbers; the ten binding sites of Section 4 are illustrative sequences resembling the AP-1 site, typed in; expression rates, fold changes and allele counts are simulated with known truth. Prior strengths and test characteristics are illustrative.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Write** the binomial, Poisson and normal distributions in exponential-family form $h(x)e^{\eta(\theta)\cdot T(x) - B(\theta)}$,
   **identify** the sufficient statistic, and **explain** why the family is useful.
2. **Apply** Bayes' theorem to parameters with a known prior (screening test; drawer of coins) and **build** a Bayes table
   with prior, likelihood, numerator and posterior.
3. **Contrast** the frequentist and Bayesian schools: what a probability means, what is random, and what each uses.
4. **Compute** pseudocount estimates and **relate** them to the Beta prior; **derive** the Beta posterior
   $\text{Beta}(k + \alpha, n - k + \beta)$.
5. **Derive** MAP estimates for the Bernoulli (Beta prior), Poisson (Gamma prior) and normal mean (normal prior), and
   **compare** MLE, MAP and posterior mean.
6. **Interpret** the normal–normal MAP as a precision-weighted average that **shrinks** noisy estimates toward the prior.
7. **Quantify** by simulation when a prior helps (small samples, sensible prior) and when it hurts (a confident wrong prior).
8. **Recognise** priors in biology: pseudocounts in motif matrices, rare variants, empirical-Bayes shrinkage of expression
   estimates, and ridge regression.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports | 3 min |
| 1 · The exponential family | CLT recap; binomial, Poisson, normal in one form; sufficient statistics | 25 min |
| 2 · Bayes' theorem with a known prior | screening test; a drawer of three coin types; MAP vs MLE | 20 min |
| 3 · Frequentists and Bayesians | the meaning of probability; the two paths from Bayes' formula; the Bayesian method | 10 min |
| 4 · Pseudocounts | 100 tosses, 0 heads; strong vs weak beliefs; motif matrices | 15 min |
| 5 · The Beta distribution | density, Beta and Gamma functions, shapes, mean and mode | 15 min |
| 6 · The Beta posterior | conjugacy derived; prior → posterior pictures; credible intervals | 15 min |
| 7 · Maximum a posteriori estimation | definition; Beta-prior MAP derived; a discrete prior; MLE vs MAP vs posterior mean; rare variants | 25 min |
| 8 · Conjugate priors; Poisson with a Gamma prior | hyperparameters; MAP derived; empirical-Bayes shrinkage of expression rates | 20 min |
| 9 · The normal mean with a normal prior | precision-weighted averages; shrinking fold changes; ridge | 20 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A transcription factor's binding motif is built from ten experimentally confirmed binding sites. At position 4 the sites
show C seven times and G three times — never A, never T. Maximum likelihood (Module 23) estimates $P(\text{T at position 4}) =
0/10 = 0$, so *any* DNA sequence with a T there gets probability exactly zero of being a binding site: one unseen letter vetoes
the whole site. Nobody believes that ten sites prove a base is impossible.

The same zero haunts population genetics (a variant never seen in 50 genomes is not absent from the population), surveillance
(no resistant isolate among the first 100 tested does not mean resistance is impossible) and single-cell biology (a gene
with zero molecules in three cells of a rare cell type is not necessarily silent). In each case we know *something* before the
experiment — four bases exist; variants of every frequency exist; most genes are expressed at moderate levels. **Bayesian
estimation** puts that knowledge into a **prior distribution** and combines it with the likelihood by Bayes' theorem:

$$\underbrace{p(\theta\mid\mathcal D)}_{\text{posterior}} \;\propto\; \underbrace{p(\mathcal D\mid\theta)}_{\text{likelihood}}\;\times\;\underbrace{p(\theta)}_{\text{prior}} .$$

*In words:* what we believe about a parameter after the experiment is what the data say, weighted by what we believed before.
The peak of the posterior is the **maximum a posteriori (MAP)** estimate.

| Biological question | Tool | Where |
|---|---|---|
| Probability that a patient with a positive screening test is ill | Bayes' theorem with a known prior | 2 |
| Which of three known strains/coin types produced these data? | Bayes table, MAP vs MLE | 2 |
| A base never seen at a motif position | pseudocounts (Laplace smoothing) | 4 |
| Frequency of a resistance gene or rare variant seen 0 times | Beta prior, posterior, credible bound | 6, 7 |
| Expression rate of a gene from three cells | Gamma–Poisson MAP, empirical Bayes | 8 |
| Log fold change from three noisy replicates | normal prior → shrinkage (DESeq2-style) | 9 |

> 🧪 **Simulated data, with a known truth.** The coin, screening-test and discrete-prior examples are classic small worked
> examples with typed-in numbers; the binding sites are illustrative typed-in sequences; every other data set is simulated
> with known parameters, so we can measure — over thousands of simulated studies — whether a prior improved the estimate.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sympy", "sympy")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sympy as sp
from scipy import special, stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

---
## 1 · The exponential family: one form for (almost) every distribution so far

> 📍 **Where we are.** In Module 22 the **central limit theorem** showed that, for i.i.d. $X_1,\dots,X_n$ with mean $\mu$ and
> SD $\sigma$, the sum $S_n$ and the mean $\bar X_n$ have $E(S_n) = n\mu$, $\operatorname{Var}(S_n) = n\sigma^2$,
> $E(\bar X_n) = \mu$, $\operatorname{Var}(\bar X_n) = \sigma^2/n$, the same standardisation $Z_n = (S_n - n\mu)/(\sigma\sqrt n) =
> (\bar X_n - \mu)/(\sigma/\sqrt n)$, and, for large $n$, $\bar X_n\approx\mathcal N(\mu, \sigma^2/n)$, $S_n\approx\mathcal N(n\mu, n\sigma^2)$,
> $Z_n\approx\mathcal N(0,1)$. We use the CLT because the exact distribution of a sum is usually unfamiliar or too costly to
> compute, and the CLT applies to almost any distribution (some need a larger $n$). In Module 23 we estimated parameters by
> maximum likelihood, deriving the MLE separately for each distribution. Before adding priors, we notice that all those
> distributions share one algebraic form — and that form is what makes the Bayesian calculations of this module work.

**Probability distributions so far:** Bernoulli, binomial, geometric, Poisson, negative binomial, normal, exponential (and
others, such as the gamma and beta). All are special cases of one general class of models, the **exponential family**.

**Definition.** A single-parameter **exponential family** is a parametric set of probability distributions whose PDF or PMF can
be written as

$$P(X\mid\theta) = h(X)\,e^{\eta(\theta)\cdot T(X) - B(\theta)} ,$$

where $\eta(\theta)$, $B(\theta)$, $T(X)$ and $h(X)$ are known functions, $h(X)\ge 0$, and $T(X)$ is called the **sufficient
statistic**.

*In words:* the probability splits into a part that depends only on the data ($h$), a part that depends only on the parameter
($B$), and a single interaction term in which the parameter (through $\eta$, the **natural parameter**) multiplies a summary of
the data ($T$). With several parameters, $\eta$ and $T$ become vectors and $\eta\cdot T$ is a dot product (Module 07). (Many
books write the same thing as $h(x)\exp(\boldsymbol\eta^\top T(x) - A(\boldsymbol\eta))$, with $A$ the *log-partition function*.)

🧮 **Math refresher — the three rules we need.** $e^{\log x} = x$ (so any positive quantity can be written as an exponential);
$e^a e^b = e^{a+b}$; and $\log(MN) = \log M + \log N$, $\log(M/N) = \log M - \log N$, $\log M^N = N\log M$. Also
$(a - b)^2 = a^2 - 2ab + b^2$.

**The binomial** $X\sim\text{Binomial}(n, p)$ with $n$ fixed:

$$\begin{aligned}
P(X = x) &= \binom nx\,p^x(1-p)^{n-x} = \binom nx\,e^{\log[p^x(1-p)^{n-x}]} = \binom nx\,e^{x\log p + (n-x)\log(1-p)}\\
&= \binom nx\,e^{x\log p + n\log(1-p) - x\log(1-p)} = \underbrace{\binom nx}_{h(x)}\,\exp\Big(\underbrace{x}_{T(x)}\,\underbrace{\log\frac{p}{1-p}}_{\eta(p)} - \underbrace{\big(-n\log(1-p)\big)}_{B(p)}\Big) .
\end{aligned}$$

*In words:* write the powers as an exponential of a logarithm, expand, and collect the terms multiplying $x$. The natural
parameter of the binomial is the **log-odds** $\log\frac{p}{1-p}$ (the *logit*), and the sufficient statistic is the number of
successes.

**The Poisson** $X\sim\text{Poisson}(\lambda)$, $x = 0, 1, 2, \dots$:

$$P(X = x) = e^{-\lambda}\frac{\lambda^x}{x!} = \frac{1}{x!}\,e^{-\lambda}e^{\log\lambda^x} = \frac{1}{x!}\,e^{\log\lambda^x - \lambda}
= \underbrace{\frac{1}{x!}}_{h(x)}\exp\Big(\underbrace{x}_{T(x)}\,\underbrace{\log\lambda}_{\eta(\lambda)} - \underbrace{\lambda}_{B(\lambda)}\Big) .$$

*In words:* the natural parameter is the **log-rate** and the sufficient statistic is the count itself.

**The normal** $X\sim\mathcal N(\mu, \sigma^2)$ (two parameters, so $\eta$ and $T$ are vectors). Expand the square, then pull
$\frac{1}{\sigma} = e^{-\log\sigma}$ into the exponent:

$$\begin{aligned}
P(x) &= \frac{1}{\sqrt{2\pi\sigma^2}}e^{-\frac{1}{2\sigma^2}(x-\mu)^2} = \frac{1}{\sqrt{2\pi\sigma^2}}e^{-\frac{1}{2\sigma^2}(x^2 - 2\mu x + \mu^2)}
= \frac{1}{\sqrt{2\pi\sigma^2}}e^{-\frac{1}{2\sigma^2}x^2 + \frac{\mu}{\sigma^2}x - \frac{\mu^2}{2\sigma^2}}\\
&= \underbrace{\frac{1}{\sqrt{2\pi}}}_{h(x)}\exp\Big(\underbrace{\Big[-\frac{1}{2\sigma^2},\ \frac{\mu}{\sigma^2}\Big]}_{\boldsymbol\eta(\theta)}\cdot\underbrace{\begin{bmatrix}x^2\\ x\end{bmatrix}}_{T(x)}
- \underbrace{\Big(\frac{\mu^2}{2\sigma^2} + \log\sigma\Big)}_{B(\theta)}\Big) .
\end{aligned}$$

*In words:* the sufficient statistic of the normal is the pair $(x^2, x)$ and the natural parameters are $-1/(2\sigma^2)$ and
$\mu/\sigma^2$; the row-times-column product $[a\ b]\begin{bmatrix}c\\ d\end{bmatrix} = ac + bd$ (Module 08) reproduces the two middle terms.
Note the sign: since the exponent is $\eta\cdot T - B$, the leftover terms $-\frac{\mu^2}{2\sigma^2} - \log\sigma$ are $-B(\theta)$,
so $B(\theta) = \frac{\mu^2}{2\sigma^2} + \log\sigma$. Let SymPy confirm all three rewrites.

In [ ]:
# ── 1.1 SymPy: each exponential-family form equals the original PMF/PDF ──
x, n, p, lam, mu, sig = sp.symbols("x n p lambda mu sigma", positive=True)
checks = {
    "binomial": (sp.binomial(n, x) * p ** x * (1 - p) ** (n - x),
                 sp.binomial(n, x) * sp.exp(x * sp.log(p / (1 - p)) - (-n * sp.log(1 - p)))),
    "Poisson": (sp.exp(-lam) * lam ** x / sp.factorial(x),
                1 / sp.factorial(x) * sp.exp(x * sp.log(lam) - lam)),
    "normal": (1 / sp.sqrt(2 * sp.pi * sig ** 2) * sp.exp(-(x - mu) ** 2 / (2 * sig ** 2)),
               1 / sp.sqrt(2 * sp.pi) * sp.exp((-1 / (2 * sig ** 2)) * x ** 2 + (mu / sig ** 2) * x
                                               - (mu ** 2 / (2 * sig ** 2) + sp.log(sig)))),
}
for name, (original, family_form) in checks.items():
    ratio = sp.simplify(sp.expand_log(sp.log(original) - sp.log(family_form), force=True))
    print(f"{name:9s}: log(original) − log(exponential-family form) simplifies to {ratio}")
# numerical spot check at arbitrary values
vals = {x: 3, n: 10, p: sp.Rational(3, 10), lam: sp.Rational(5, 2), mu: 1, sig: 2}
for name, (original, family_form) in checks.items():
    print(f"{name:9s}: {float(original.subs(vals)):.6f} = {float(family_form.subs(vals)):.6f}")

🔍 **What just happened?** For all three distributions the difference of the logarithms simplifies to 0, and plugging in numbers
(binomial with $n = 10$, $p = 0.3$, $x = 3$; Poisson with $\lambda = 2.5$; normal with $\mu = 1$, $\sigma = 2$) gives identical
values (0.266828, 0.213763 and 0.120985): the rewritten forms are the same functions, only organised differently.

### 1.2 Why the form matters: sufficient statistics

For i.i.d. data the likelihood of an exponential-family model is a product of $n$ such factors:

$$p(\mathcal D\mid\theta) = \prod_{i=1}^n h(x_i)\,e^{\eta(\theta)T(x_i) - B(\theta)} = \Big[\prod_{i=1}^n h(x_i)\Big]\,
\exp\Big(\eta(\theta)\sum_{i=1}^n T(x_i) - nB(\theta)\Big) .$$

*In words:* the parameter meets the data only through the single number $\sum_i T(x_i)$ (and $n$). Two data sets with the same
$n$ and the same $\sum_i T(x_i)$ have likelihoods that differ by a constant factor, so they lead to exactly the same MLE — and,
in Bayesian analysis, to exactly the same posterior. That is what **sufficient** means: once you know $\sum_i T(x_i)$, the rest
of the data carries no further information about $\theta$. For Poisson counts it is the total count; for coin flips the number
of heads; for normal data the pair $(\sum x_i^2, \sum x_i)$.

In [ ]:
# ── 1.2 Two different sets of Poisson counts with the same total: parallel log-likelihoods, same MLE ──
counts_a = np.array([3, 0, 2, 5, 0])          # e.g. molecules in five cells
counts_b = np.array([2, 2, 2, 2, 2])          # same n = 5 and same total 10
lam_grid = np.linspace(0.2, 6, 300)
ll_a = np.array([stats.poisson.logpmf(counts_a, l).sum() for l in lam_grid])
ll_b = np.array([stats.poisson.logpmf(counts_b, l).sum() for l in lam_grid])
diff = ll_a - ll_b
print(f"totals: {counts_a.sum()} and {counts_b.sum()};  MLEs: {lam_grid[ll_a.argmax()]:.3f} and {lam_grid[ll_b.argmax()]:.3f}")
print(f"ℓ_a(λ) − ℓ_b(λ) over the whole grid: min {diff.min():.6f}, max {diff.max():.6f}  (a constant = log Π h)")
print(f"log Π h(x_i) difference = Σ log(1/x_i!) for a minus for b = "
      f"{-np.sum(special.gammaln(counts_a + 1)) + np.sum(special.gammaln(counts_b + 1)):.6f}")

🔍 **What just happened?** The counts 3, 0, 2, 5, 0 and 2, 2, 2, 2, 2 look very different, but both total 10 in 5 cells. Their
log-likelihood curves differ by the same constant (−3.8067) at every $\lambda$ — exactly $\log\prod h(x_i)$ for one set minus
the other — so both peak at $\hat\lambda = 2$. Under a Poisson model the *pattern* of the counts says nothing about $\lambda$
(it could tell us the Poisson model is wrong — Module 23, Section 2 — but that is a different question).

**Why is the exponential family useful?**
* The **sufficient statistic is explicitly represented** in the form of the distribution: read off $T(x)$ and you know what
  to compute from the data.
* Once statistical methods and theories are developed for an exponential family, **they can be applied to all members**: one
  derivation of the MLE ($\frac{d}{d\theta}$ of $\eta(\theta)\sum T(x_i) - nB(\theta)$), one theory of conjugate priors
  (Section 8), one theory of generalised linear models.

Not everything is in the family: the Uniform$(0, \theta)$ of Module 23 is not, because its support depends on $\theta$ — and
indeed it behaved differently (a boundary MLE).

### 🌉 Bridge — natural parameters are the links of generalised linear models

The natural parameter of the Bernoulli/binomial is the logit $\log\frac{p}{1-p}$; that of the Poisson is $\log\lambda$.
**Logistic regression** models the logit as a linear function of predictors, $\log\frac{p}{1-p} = \beta_0 + \beta_1 x$
([BiologyToML 09](../../../../ml/tutorial/BiologyToML/notebooks/09_logistic_regression.ipynb)); **Poisson and negative-binomial regression** for RNA-seq counts model
$\log\lambda$ linearly ([BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb)). Both are generalised linear
models, and both owe their tidy, concave likelihoods to the exponential-family form.

✏️ **Your turn.** (a) Write the Bernoulli PMF $p^x(1-p)^{1-x}$ and the exponential density $\lambda e^{-\lambda x}$ in
exponential-family form: identify $h$, $T$, $\eta$ and $B$. (b) Using $\sum_i T(x_i)$, write the MLE equation for the Poisson in
one line: $\frac{d}{d\lambda}[\log\lambda\sum_i x_i - n\lambda] = 0$.

### 📘 Lecture notes — the exponential family

**Distributions so far** — Bernoulli, binomial, geometric, Poisson, negative binomial, normal, exponential, and others — are
all special cases of the **exponential family**:

$$P(X\mid\theta) = h(X)\,e^{\eta(\theta)\cdot T(X) - B(\theta)}, \qquad h(X)\ge 0,\quad T(X) = \text{sufficient statistic} .$$

| Distribution | $h(x)$ | $T(x)$ | $\eta(\theta)$ | $B(\theta)$ |
|---|---|---|---|---|
| Binomial($n, p$), $n$ fixed | $\binom nx$ | $x$ | $\log\frac{p}{1-p}$ | $-n\log(1-p)$ |
| Poisson($\lambda$) | $\frac{1}{x!}$ | $x$ | $\log\lambda$ | $\lambda$ |
| Normal($\mu, \sigma^2$) | $\frac{1}{\sqrt{2\pi}}$ | $(x^2, x)$ | $\big(-\frac{1}{2\sigma^2}, \frac{\mu}{\sigma^2}\big)$ | $\frac{\mu^2}{2\sigma^2} + \log\sigma$ |

*In words:* rewrite each PMF/PDF with $e^{\log(\cdot)}$, expand the logarithms, and group the terms that multiply the data.

**Why useful.** (1) The sufficient statistic is explicit: the likelihood of i.i.d. data depends on the data only through
$\sum_i T(x_i)$ (Poisson counts 3, 0, 2, 5, 0 and 2, 2, 2, 2, 2 give the same $\hat\lambda = 2$). (2) Methods and theory developed
once apply to every member (MLE, conjugate priors, GLMs).

**Common confusion.** In the normal form the constant terms $-\frac{\mu^2}{2\sigma^2} - \log\sigma$ in the exponent are $-B(\theta)$,
not $B(\theta)$. Uniform$(0,\theta)$ is *not* in the family (its support depends on $\theta$).

> 🎓 **Exam tip.** To show a distribution is in the family: (1) write it as $e^{\log(\cdot)}$, (2) expand the log, (3) collect
> what multiplies the data. Label $h, T, \eta, B$ explicitly — that is what is graded.

---
## 2 · Bayes' theorem with a known prior

> 📍 **Where we are.** Module 17 introduced Bayes' theorem for events; Module 23 met it again, and then dropped the prior to
> get the MLE. Here we keep it — first in the easy case where everybody agrees what the prior is.

**Bayes' theorem.**

$$P(\mathcal H\mid\mathcal D) = \frac{P(\mathcal D\mid\mathcal H)\,P(\mathcal H)}{P(\mathcal D)}, \qquad\text{i.e.}\qquad
P(\text{hypothesis}\mid\text{data}) = \frac{P(\text{data}\mid\text{hypothesis})\,P(\text{hypothesis})}{P(\text{data})} ,$$

*In words:* $P(\mathcal H\mid\mathcal D)$ is the **posterior**, $P(\mathcal D\mid\mathcal H)$ the **likelihood**, $P(\mathcal H)$ the
**prior**, and $P(\mathcal D)$ — the overall probability of the data, sometimes called the data distribution or **evidence** — is
the normaliser, $P(\mathcal D) = \sum_{\phi}P(\mathcal D\mid\phi)P(\phi)$ over all hypotheses $\phi$ (law of total probability,
Module 17). When all the probabilities on the right are known exactly, we can compute the probability on the left exactly.

> ⚠️ **Common confusion.** "The data distribution" in Bayes' formula is $P(\mathcal D)$, the denominator — not the likelihood
> $P(\mathcal D\mid\mathcal H)$. The likelihood is a different number for every hypothesis; $P(\mathcal D)$ is one number that
> makes the posteriors add up to 1.

**Example: a screening test.** A screening test for a disease is both sensitive and specific: the true-positive rate is 99 %,
the false-positive rate 2 %. The disease affects 0.5 % of the population. A random person tests positive; what is the
probability that they have the disease? Hypothesis $\mathcal H$ = "the person has the disease"; data $\mathcal D$ = "the test was
positive".

$$P(\mathcal H\mid\mathcal D) = \frac{P(\mathcal D\mid\mathcal H)P(\mathcal H)}{P(\mathcal D)} = \frac{(0.99)(0.005)}{(0.99)(0.005) + (0.02)(0.995)} = 0.199 \approx 20\,\% .$$

*In words:* among positives, true cases (0.495 % of everyone) are outnumbered four to one by false alarms from the healthy
majority (1.99 % of everyone). This works perfectly *because the prior — the prevalence 0.5 % — was known*.

In [ ]:
# ── 2.1 The screening test: Bayes' theorem, and a check by simulating one million people ──
prevalence, sensitivity, false_pos = 0.005, 0.99, 0.02
posterior = sensitivity * prevalence / (sensitivity * prevalence + false_pos * (1 - prevalence))
print(f"P(disease | positive) = {sensitivity * prevalence:.5f} / ({sensitivity * prevalence:.5f} + "
      f"{false_pos * (1 - prevalence):.5f}) = {posterior:.4f}")
rng_s = np.random.default_rng(SEED + 21)
ill = rng_s.random(1_000_000) < prevalence
positive = np.where(ill, rng_s.random(1_000_000) < sensitivity, rng_s.random(1_000_000) < false_pos)
print(f"simulation of 1,000,000 people: {positive.sum()} positives, of whom {np.sum(ill & positive)} are ill "
      f"→ {np.sum(ill & positive) / positive.sum():.4f}")

🔍 **What just happened?** The formula gives 0.1992, and a simulated population of one million people agrees (24,849 positives, of whom 4,934 are ill: 0.1986):
only about one positive in five is ill.

### 2.2 A drawer of coins: hypotheses about a parameter

There are three types of coins with different probabilities of landing heads: type A coins are fair (0.5), type B are bent
(0.6), type C are bent (0.9). A drawer contains **10 coins: 5 of type A, 3 of type B and 2 of type C**. You pick a coin at
random; without showing it to you, someone flips it once and gets **tails**. What is the probability that it is type A? B? C?

Now the "hypothesis" is the value of the parameter $\theta$ = probability of heads, $\theta\in\{0.5, 0.6, 0.9\}$, and the drawer
gives a known prior $p(\theta) = 0.5, 0.3, 0.2$. (Biological twin: a patient's genotype at a drug-metabolism gene is one of
three types with known population frequencies and known response probabilities; one observed outcome updates the probability
of each genotype.) We organise the computation as a **Bayes table**.

In [ ]:
# ── 2.2 Bayes tables for the drawer of 10 coins: one tails, then 6 heads and 4 tails ──
def bayes_table(hypotheses, prior, likelihood):
    """Bayes table: prior, likelihood, Bayes numerator p(D|θ)p(θ) and posterior p(θ|D) = numerator / Σ numerators."""
    numerator = np.asarray(prior) * np.asarray(likelihood)
    return pd.DataFrame({"prior p(θ)": prior, "likelihood p(D|θ)": likelihood,
                         "numerator p(D|θ)·p(θ)": numerator, "posterior p(θ|D)": numerator / numerator.sum()},
                        index=pd.Index(hypotheses, name="hypothesis θ"))


types = ["A (0.5)", "B (0.6)", "C (0.9)"]
p_heads = np.array([0.5, 0.6, 0.9])
prior_drawer = np.array([5, 3, 2]) / 10
one_tail = bayes_table(types, prior_drawer, 1 - p_heads)
print("D = one tails:"); print(one_tail.round(3).to_string())
print(f"P(D) = Σ numerators = {one_tail['numerator p(D|θ)·p(θ)'].sum():.2f}")
ten_flips = bayes_table(types, prior_drawer, p_heads ** 6 * (1 - p_heads) ** 4)
print("\nD = 6 heads and 4 tails (same coin, 10 flips in total):")
print(ten_flips.to_string(float_format=lambda v: f"{v:.3e}" if v < 0.01 else f"{v:.3f}"))
print(f"\nMAP (largest posterior): {ten_flips['posterior p(θ|D)'].idxmax()} | "
      f"MLE (largest likelihood): {ten_flips['likelihood p(D|θ)'].idxmax()}")
# sequential updating: yesterday's posterior is today's prior
step2 = bayes_table(types, one_tail["posterior p(θ|D)"].to_numpy(), p_heads ** 6 * (1 - p_heads) ** 3)
print("sequential (posterior after 1 tails, then 6 H + 3 T):", step2["posterior p(θ|D)"].round(3).tolist())

🔍 **What just happened?** After one tails the numerators are $0.5\times0.5 = 0.25$, $0.3\times0.4 = 0.12$ and
$0.2\times0.1 = 0.02$; they sum to $P(\mathcal D) = 0.39$, so the posteriors are $0.25/0.39 = 0.641$, $0.12/0.39 = 0.308$ and
$0.02/0.39 = 0.051$. Type A is the most probable: $\hat\theta_{\text{MAP}} = A$.

After 6 heads and 4 tails the likelihoods are $0.5^{10} = 0.977\times10^{-3}$, $0.6^6 0.4^4 = 1.194\times10^{-3}$ and
$0.9^6 0.1^4 = 0.053\times10^{-3}$. Type B explains the data best ($\hat\theta_{\text{MLE}} = B$), but A is more common in the
drawer, and the posterior still favours A: 0.570 vs 0.418 vs 0.012 ($\hat\theta_{\text{MAP}} = A$). The prior (5 vs 3 coins)
outweighs a likelihood ratio of only $1.194/0.977 = 1.22$. Notice that the prior column did not change — we combined the
original prior with all ten flips — and that updating step by step (the posterior after one tails becomes the prior for the
next nine flips) gives exactly the same answer.

> ⚠️ **Check your arithmetic.** The posteriors must sum to 1 and each must be the numerator divided by the *sum* of the
> numerators: for one tails 0.641, 0.308, 0.051 (sum 1.000). A table whose posteriors do not add to 1, or that divides by the
> wrong total, is wrong.

✏️ **Your turn.** (a) The coin is flipped once more and shows heads. Starting from the one-tails posterior, update by hand.
(b) Which drawer composition would make B the MAP after 6 heads and 4 tails? Find the smallest number of type-B coins (out of
10, with C fixed at 2) for which it happens.

In [ ]:
# ── 2.3 Flip after flip: the drawer posterior when the chosen coin is really type B (SIMULATED flips) ──
rng_d = np.random.default_rng(SEED + 23)
flips_b = (rng_d.random(1000) < 0.6).astype(int)             # 1,000 flips of a type-B coin (P(heads) = 0.6)
heads_so_far = np.concatenate([[0], np.cumsum(flips_b)])
n_so_far = np.arange(len(flips_b) + 1)
log_numerator = (np.log(prior_drawer)[None, :] + heads_so_far[:, None] * np.log(p_heads)[None, :]
                 + (n_so_far - heads_so_far)[:, None] * np.log(1 - p_heads)[None, :])   # logs avoid underflow
post_path = np.exp(log_numerator - log_numerator.max(axis=1, keepdims=True))
post_path /= post_path.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for j, (name, col) in enumerate(zip(types, ["tab:gray", "tab:blue", "tab:red"])):
    ax.plot(n_so_far, post_path[:, j], color=col, label=f"type {name}")
ax.set_xlabel("number of flips of the chosen coin"); ax.set_ylabel("posterior probability"); ax.legend()
ax.set_title("SIMULATED: a type-B coin — enough data overrule the prior 5 : 3 : 2")
plt.tight_layout(); plt.show()
b_is_map = post_path.argmax(axis=1) == 1
b_for_good = np.flatnonzero(~b_is_map).max() + 1               # B is the MAP at every flip from here on
for m in [10, 100, 300, 1000]:
    print(f"after {m:4d} flips ({heads_so_far[m]} heads): posterior (A, B, C) = {post_path[m].round(4).tolist()}")
print(f"B is the MAP at every flip from flip {b_for_good} on; posterior of B first exceeds 0.95 at flip "
      f"{np.flatnonzero(post_path[:, 1] > 0.95)[0]} and 0.99 at flip {np.flatnonzero(post_path[:, 1] > 0.99)[0]}")

🔍 **What just happened?** The chosen coin is really type B. At the start the posterior is the prior (0.5, 0.3, 0.2); the
first few flips make it wobble, and type C, which needs about 90 % heads, is ruled out quickly. Telling A (0.5) from B (0.6)
takes much longer: after 10 flips (5 heads) A is still favoured (0.670 vs 0.328), after 100 flips (56 heads) the two are
nearly tied (0.530 vs 0.470), B is the MAP at every flip only from flip 159 on, its posterior first exceeds 0.95 at flip 357
and 0.99 at flip 484, and after 1,000 flips it is 1.000. A known prior is a head start, not a verdict: with enough data the likelihood decides.

---
## 3 · The frequentist and the Bayesian schools

> 📍 **Where we are.** With a known prior nobody argues: everyone follows Bayes' rule as written. The argument starts when
> the prior is *not* known — which is almost always.

Both schools of statistics start from probability. For Bayesian inference we take $H$ to be a hypothesis (parameters) and $D$
some data. Different people will have different *a priori* beliefs, but we would still like to make useful inferences from
data. When $p(H)$ is known, there is no disagreement.

**When the prior is not known**, Bayesian logic requires us to develop a prior from the information we have (experience,
previous studies, intuition), while **frequentist** logic uses only the information in the provided data. The reasons for this
split are both practical (ease of implementation and computation) and philosophical (subjectivity versus objectivity, and the
nature of probability). The main philosophical difference concerns **the meaning of probability**:

| | Frequentists | Bayesians |
|---|---|---|
| What a probability is | the long-run frequency of a repeatable random experiment | a degree of belief, or state of knowledge, about a proposition |
| Interpretation | objective | subjective |
| "A coin has probability ½ of tails" means | the relative frequency of tails goes to ½ as the number of flips goes to infinity (Module 22's LLN) | you "believe" you will get tails 50 % of the time |
| The parameter $H$ | fixed and not random (if unknown) | random: a range of values, each with its own probability of being true |
| Uses | the likelihood only, $L(H\mid D) = p(D\mid H)$ | prior + likelihood → posterior |
| Typical output | MLE, confidence interval, p-value | posterior distribution, MAP or posterior mean, credible interval |

In practice there is **no universally accepted prior** — which is the frequentists' main objection, and the reason Bayesians
work hard on sensible defaults and on checking how much conclusions depend on the prior.

In [ ]:
# ── 3.1 Two paths from Bayes' formula ──
fig, ax = plt.subplots(figsize=(9, 4.0))
ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis("off")
style = dict(ha="center", va="center", fontsize=10)
ax.text(5, 4.3, r"$P(H\mid D) = \dfrac{P(D\mid H)\,P(H)}{P(D)}$", bbox=dict(boxstyle="square,pad=0.5", fc="0.95"), **style)
ax.text(8.6, 4.3, "everyone uses Bayes' formula\nwhen the prior P(H) is known", fontsize=8, ha="center", va="center")
ax.text(0.9, 4.3, "probability\n(mathematics)", fontsize=9, weight="bold", ha="center", va="center")
ax.text(0.9, 1.4, "statistics\n(art)", fontsize=9, weight="bold", ha="center", va="center")
ax.annotate("", xy=(2.9, 1.9), xytext=(4.6, 3.8), arrowprops=dict(arrowstyle="-|>", color="tab:cyan", lw=2))
ax.annotate("", xy=(7.1, 1.9), xytext=(5.4, 3.8), arrowprops=dict(arrowstyle="-|>", color="tab:blue", lw=2))
ax.text(3.2, 3.0, "Bayesian path", color="tab:cyan", fontsize=10)
ax.text(6.2, 3.0, "frequentist path", color="tab:blue", fontsize=10)
ax.text(2.9, 1.4, r"$P_{\mathrm{posterior}}(H\mid D) = \dfrac{P(D\mid H)\,P_{\mathrm{prior}}(H)}{P(D)}$",
        bbox=dict(boxstyle="square,pad=0.5", fc="#e0f3f8"), **style)
ax.text(7.3, 1.4, r"likelihood  $L(H; D) = P(D\mid H)$", bbox=dict(boxstyle="square,pad=0.5", fc="#deebf7"), **style)
ax.text(2.9, 0.45, "Bayesians require a prior, so they develop\none from the best information they have", fontsize=8, **{k: v for k, v in style.items() if k != "fontsize"})
ax.text(7.3, 0.45, "without a known prior, frequentists draw\ninferences from the likelihood alone", fontsize=8, **{k: v for k, v in style.items() if k != "fontsize"})
ax.set_title("In practice there is no universally accepted prior", color="darkred")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The diagram summarises the split. Top: probability theory — Bayes' formula, which everyone uses
when the prior is known (the screening test, the drawer). Bottom: statistics, where the prior is usually unknown. The Bayesian
path supplies a prior and computes a posterior; the frequentist path keeps only the likelihood (Module 23's MLE, Module 22's
confidence intervals, Module 25's tests).

**The Bayesian method.** Bayesian inference is usually carried out in three steps:

1. Choose a probability density $f(\theta)$ — the **prior distribution** — that expresses our beliefs about a parameter $\theta$
   before we see any data.
2. Choose a statistical model $f(x\mid\theta)$ that reflects our beliefs about $x$ given $\theta$. (We now write $f(x\mid\theta)$,
   conditioning on $\theta$, instead of $f(x;\theta)$, because $\theta$ is a random variable.)
3. After observing data $X_1,\dots,X_n$, update our beliefs and calculate the **posterior distribution**
   $f(\theta\mid X_1,\dots,X_n)\propto f(X_1,\dots,X_n\mid\theta)\,f(\theta)$.

Sections 4–9 carry out these steps for coins, counts and continuous measurements.

### 📘 Lecture notes — Bayes' theorem with a known prior; frequentists vs Bayesians

**Bayes' theorem.** $P(\mathcal H\mid\mathcal D) = \frac{P(\mathcal D\mid\mathcal H)P(\mathcal H)}{P(\mathcal D)}$: posterior,
likelihood, prior, and $P(\mathcal D) = \sum_\phi P(\mathcal D\mid\phi)P(\phi)$ (the data distribution / evidence). Known
right-hand side ⇒ exact posterior. *Screening:* sensitivity 0.99, false-positive rate 0.02, prevalence 0.005 ⇒
$P(\text{ill}\mid +) = 0.00495/(0.00495 + 0.0199) = 0.199\approx 20\,\%$.

**Known-prior example (drawer of 10 coins: 5 A, 3 B, 2 C; P(heads) = 0.5, 0.6, 0.9).** One tails: numerators 0.25, 0.12, 0.02
(sum 0.39) → posterior **0.641, 0.308, 0.051** → MAP = A. Six heads, four tails: likelihoods $0.977, 1.194, 0.053\times10^{-3}$;
numerators $4.88, 3.58, 0.11\times10^{-4}$; posterior **0.570, 0.418, 0.012** → MAP = A but MLE = B. The prior column does
not change; sequential updating gives the same result.

**Schools.** Frequentist: $H$ fixed, not random; uses the likelihood only, $L(H\mid D) = p(D\mid H)$; probabilities are long-run
frequencies (objective); relies only on observed data. Bayesian: $H$ random with a prior; prior + likelihood → posterior;
probability is a degree of belief (subjective). When the prior is known there is no disagreement; in practice there is no
universally accepted prior.

**Bayesian method.** (1) Prior $f(\theta)$; (2) model $f(x\mid\theta)$; (3) posterior $f(\theta\mid X_1,\dots,X_n)$.

> 🎓 **Exam tip.** In a Bayes table always show the numerator column and its sum $P(\mathcal D)$, and check that the
> posteriors add to 1. The MAP is the row with the largest posterior; the MLE is the row with the largest likelihood.

---
## 4 · Pseudocounts: adding imaginary data

> 📍 **Where we are.** The simplest way to put a belief into an estimate needs no new distribution at all: pretend you have
> already seen some data.

Take the coin-flip example of Module 23: $\mathcal D = X_1,\dots,X_n$ with $X_i\sim\text{Bernoulli}(\alpha)$. Summarise $\mathcal D$ by
$c_H$ and $c_T$, the counts of heads and tails. We saw that

$$\hat\alpha_{\text{MLE}} = \frac{c_H}{c_H + c_T} .$$

*In words:* the fraction of heads. But this assumes we know nothing about $\alpha$ ahead of time. If we believe the coin is
roughly 50/50, we can add **pseudocounts** $c_{H_0}$ and $c_{T_0}$ — imaginary heads and tails — to the observed counts:

$$\hat\alpha_{\text{MLE-PC}} = \frac{c_H + c_{H_0}}{c_H + c_T + c_{H_0} + c_{T_0}} .$$

*In words:* the fraction of heads among real *and* imaginary flips. Equal pseudocounts pull the estimate toward ½; their total
says how hard.

**Example.** A coin is thrown 100 times: $c_H = 0$, $c_T = 100$. Vanilla MLE says $\hat\alpha = 0/(0 + 100) = 0$. With a small belief
that the coin is fair, $c_{H_0} = c_{T_0} = 1$: $\hat\alpha = (0 + 1)/(0 + 100 + 1 + 1) = 1/102$. With a strong belief,
$c_{H_0} = c_{T_0} = 100$: $\hat\alpha = (0 + 100)/(0 + 100 + 100 + 100) = 100/300 = 1/3$.

In [ ]:
# ── 4.1 Pseudocount estimates: the 100-toss example, and how the data take over as n grows (SIMULATED, true α = 0.2) ──
def pseudocount_estimate(c_heads, c_tails, pc_heads, pc_tails):
    """(c_H + c_H0) / (c_H + c_T + c_H0 + c_T0): the fraction of heads among real and imaginary flips."""
    return (c_heads + pc_heads) / (c_heads + c_tails + pc_heads + pc_tails)


for pc in [0, 1, 100]:
    est = pseudocount_estimate(0, 100, pc, pc)
    print(f"c_H = 0, c_T = 100, pseudocounts c_H0 = c_T0 = {pc:3d}:  α̂ = {est:.4f}")

rng_pc = np.random.default_rng(SEED + 41)
flips = rng_pc.random(5000) < 0.2
n_seen = np.arange(1, 5001)
heads_cum = np.cumsum(flips)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for pc, col in [(0, "k"), (1, "tab:green"), (10, "tab:orange"), (100, "tab:red")]:
    est = pseudocount_estimate(heads_cum, n_seen - heads_cum, pc, pc)
    ax.semilogx(n_seen, est, color=col, lw=1.2, label=f"pseudocounts {pc} + {pc}" + (" (MLE)" if pc == 0 else ""))
    print(f"pseudocounts {pc:3d} + {pc:3d}: estimate after 10 flips = {est[9]:.3f}, after 100 = {est[99]:.3f}, "
          f"after 5000 = {est[-1]:.3f}")
ax.axhline(0.2, color="0.5", ls="--", lw=1, label="true α = 0.2")
ax.set_xlabel("number of flips n (log scale)"); ax.set_ylabel("estimate of α"); ax.legend(fontsize=7)
ax.set_title("SIMULATED: strong pseudocounts resist the data longer, but the data always win")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three estimates for 0 heads in 100 throws are 0, 1/102 = 0.0098 and 1/3 = 0.333: a weak
belief barely moves the MLE away from the impossible 0, a strong belief drags it up to 1/3 — two-thirds of the way to ½. In the simulation
(true $\alpha = 0.2$) all four estimators start in different places — after 10 flips they range from 0.100 (MLE) to 0.481 (pseudocounts
100 + 100), after 100 flips from 0.180 to 0.393 — and after 5,000 flips they lie between 0.199 and 0.210: only the strongest
prior, worth 200 imaginary flips, is still faintly visible. **Pseudocounts are a way of exerting your belief:** larger pseudocounts represent a strong prior
belief and have a greater effect on the estimate; small pseudocounts represent a weak belief and have a smaller effect; as the
sample size goes to infinity, **the data dominate the estimate**.

### 🌉 Bridge — pseudocounts in sequence motifs

A **position weight matrix** (PWM) describes a transcription-factor binding motif by the probability of each base at each
position, estimated from known binding sites. With few sites, unseen bases get probability 0 — and $\log 0 = -\infty$ vetoes
any candidate site containing one. Adding pseudocounts (often 1 per base, "Laplace smoothing", or a pseudocount split in
proportion to the genome's base composition) fixes it. Motif databases and motif-scanning tools use exactly this.

In [ ]:
# ── 4.2 A PWM from ten illustrative binding sites resembling the AP-1 site TGA(C/G)TCA (typed in) ──
sites = ["TGACTCA", "TGAGTCA", "TGACTCA", "TGACTAA", "TGAGTCA",
         "TTACTCA", "TGACTCA", "TGAGTCA", "TGACTCT", "CGACTCA"]
bases = list("ACGT")
count_matrix = pd.DataFrame([[sum(s[j] == b for s in sites) for j in range(7)] for b in bases],
                            index=bases, columns=[f"pos{j + 1}" for j in range(7)])
print("counts:"); print(count_matrix.to_string())


def position_weight_matrix(counts, pseudocount):
    """Column-wise base probabilities (count + pseudocount) / (number of sites + 4 · pseudocount)."""
    return (counts + pseudocount) / (counts.sum(axis=0) + 4 * pseudocount)


def log_odds_score(seq, pwm, background=0.25):
    """Σ_j log(P(base_j | motif) / background); −inf if a base has probability 0."""
    with np.errstate(divide="ignore"):
        return float(sum(np.log(pwm.loc[b, f"pos{j + 1}"] / background) for j, b in enumerate(seq)))


for pc in [0, 0.25, 1]:
    pwm = position_weight_matrix(count_matrix, pc)
    print(f"pseudocount {pc:4}: P(T at pos4) = {pwm.loc['T', 'pos4']:.3f}   score(TGACTCA) = {log_odds_score('TGACTCA', pwm):6.2f}"
          f"   score(TGATTCA) = {log_odds_score('TGATTCA', pwm):6.2f}   score(AAAAAAA) = {log_odds_score('AAAAAAA', pwm):6.2f}")

🔍 **What just happened?** Position 4 has C seven times and G three times. Without pseudocounts $P(\text{T at pos 4}) = 0$, so
the candidate TGATTCA — identical to the consensus except at one position — scores $-\infty$, exactly as hopeless as AAAAAAA.
With a pseudocount of 1 per base, $P(\text{T at pos 4}) = 1/14 = 0.071$, TGATTCA scores 5.24 — clearly below the
consensus (7.32) but far above AAAAAAA (−3.38), a sensible ranking. Pseudocounts turn "never seen" into "rare".

In [ ]:
# ── 4.3 The motif matrix as a picture: without and with pseudocounts ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.0), sharey=True)
base_colours = {"A": "tab:green", "C": "tab:blue", "G": "tab:orange", "T": "tab:red"}
for ax, pc in zip(axes, [0, 1]):
    pwm = position_weight_matrix(count_matrix, pc)
    bottom = np.zeros(7)
    for b in bases:
        ax.bar(range(1, 8), pwm.loc[b].to_numpy(), bottom=bottom, color=base_colours[b], edgecolor="white", label=b)
        bottom += pwm.loc[b].to_numpy()
    ax.set_xticks(range(1, 8)); ax.set_xlabel("motif position"); ax.set_title(f"pseudocount {pc} per base")
axes[0].set_ylabel("P(base | position)"); axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
fig.suptitle("Ten illustrative AP-1-like sites: base probabilities at each position", fontsize=10)
plt.tight_layout(); plt.show()
pwm0, pwm1 = position_weight_matrix(count_matrix, 0).to_numpy(), position_weight_matrix(count_matrix, 1).to_numpy()
print(f"pseudocount 0: {int(np.sum(pwm0 == 0))} of 28 entries are exactly 0, largest entry {pwm0.max():.3f}")
print(f"pseudocount 1: smallest entry {pwm1.min():.4f}, largest entry {pwm1.max():.4f}")

🔍 **What just happened?** Without pseudocounts the matrix is a set of hard rules: position 3 is A with probability 1, position
5 is T with probability 1, and 16 of the 28 entries are exactly 0. With one pseudocount per base the smallest entry becomes
$1/14 = 0.0714$ and the largest shrinks from 1 to $11/14 = 0.786$: the motif is a little blurrier, but no base is declared
impossible on the evidence of ten sites.

✏️ **Your turn.** (a) For $c_H = 3$, $c_T = 7$, compute the pseudocount estimate with $c_{H_0} = c_{T_0} = 1$ and with 10.
(b) In the PWM, use a pseudocount of 0.1. How does the score of TGATTCA change? Which pseudocount would you choose with ten
sites, and with 1,000?

---
## 5 · The Beta distribution: a distribution for a probability

> 📍 **Where we are.** Pseudocounts are a recipe. To turn them into Bayesian inference we need a *prior distribution* for a
> parameter that lives between 0 and 1 — a probability of heads, an allele frequency, a resistance rate.

🧮 **Math refresher — the Gamma function.** $\Gamma(a) = \int_0^\infty t^{a-1}e^{-t}\,dt$ extends the factorial to real numbers:
$\Gamma(a) = (a-1)!$ for positive integers ($\Gamma(1) = 1$, $\Gamma(2) = 1$, $\Gamma(3) = 2$, $\Gamma(5) = 24$) and
$\Gamma(a + 1) = a\,\Gamma(a)$. In code: `scipy.special.gamma`, or `gammaln` for its logarithm.

**The Beta distribution.** A random variable $X\in(0,1)$ has a $\text{Beta}(\alpha_S, \alpha_F)$ distribution if its density is

$$f(x;\alpha_S, \alpha_F) = \frac{x^{\alpha_S - 1}(1-x)^{\alpha_F - 1}}{B(\alpha_S, \alpha_F)}, \qquad
B(\alpha_S, \alpha_F) = \frac{\Gamma(\alpha_S)\,\Gamma(\alpha_F)}{\Gamma(\alpha_S + \alpha_F)}, \qquad 0 < x < 1 .$$

*In words:* the shape $x^{\alpha_S-1}(1-x)^{\alpha_F-1}$ looks exactly like a likelihood with $\alpha_S - 1$ successes and
$\alpha_F - 1$ failures; the **Beta function** $B(\alpha_S, \alpha_F)$ is the constant that makes the area 1. Both exponents
carry the "−1", and $B$ has the product of Gammas on top.

**Summaries.** Mean $E[X] = \dfrac{\alpha_S}{\alpha_S + \alpha_F}$; **mode** (the highest point, defined when $\alpha_S, \alpha_F > 1$)
$\dfrac{\alpha_S - 1}{\alpha_S + \alpha_F - 2}$; variance $\dfrac{\alpha_S\alpha_F}{(\alpha_S + \alpha_F)^2(\alpha_S + \alpha_F + 1)}$.
*In words:* the mean is the fraction of "successes" among $\alpha_S + \alpha_F$; the mode uses $\alpha_S - 1$ of
$\alpha_S + \alpha_F - 2$; the larger $\alpha_S + \alpha_F$, the narrower the distribution. The Beta distribution looks like an
"unbalanced normal" on $(0,1)$: symmetric when $\alpha_S = \alpha_F$, skewed otherwise; Beta(1, 1) is the uniform distribution.

In [ ]:
# ── 5.1 Beta densities, written from the formula and checked against scipy ──
def beta_density(x, a, b):
    """Beta(a, b) density x^(a−1)(1 − x)^(b−1) / B(a, b), with B(a, b) = Γ(a)Γ(b)/Γ(a + b)."""
    return x ** (a - 1) * (1 - x) ** (b - 1) / (special.gamma(a) * special.gamma(b) / special.gamma(a + b))


xg = np.linspace(0.0005, 0.9995, 2000)
dx = xg[1] - xg[0]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
rows = []
for (a_, b_), col in zip([(1, 1), (2, 2), (2, 6), (6, 2)], ["tab:blue", "tab:red", "tab:green", "tab:purple"]):
    f = beta_density(xg, a_, b_)
    ax.plot(xg, f, color=col, label=f"α_S = {a_}, α_F = {b_}")
    rows.append({"(α_S, α_F)": f"({a_}, {b_})", "B(α_S, α_F)": special.beta(a_, b_), "area": f.sum() * dx,
                 "mean (formula)": a_ / (a_ + b_), "mean (numerical)": np.sum(xg * f) * dx,
                 "mode (formula)": (a_ - 1) / (a_ + b_ - 2) if a_ + b_ > 2 else np.nan,
                 "max |ours − scipy|": np.abs(f - stats.beta.pdf(xg, a_, b_)).max()})
ax.set_xlabel("x (a probability, e.g. of heads)"); ax.set_ylabel("Beta density"); ax.set_ylim(0, 3); ax.legend()
ax.set_title("The Beta distribution on (0, 1)")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

🔍 **What just happened?** Beta(1, 1) is flat (every probability equally plausible); Beta(2, 2) is a gentle hill centred at ½;
Beta(2, 6) and Beta(6, 2) are mirror images, piled up near 0.17 and 0.83 (their modes $1/6$ and $5/6$) with means 0.25 and
0.75. Every density integrates to 1 numerically (Beta(1, 1) shows 0.9995 only because the grid stops 0.0005 short of each end), the numerical means match $\alpha_S/(\alpha_S + \alpha_F)$, and our formula agrees
with `scipy.stats.beta` to rounding error. The Beta functions are $B(1,1) = 1$, $B(2,2) = 1/6$ and $B(2,6) = B(6,2) = 1/42$.

> ⚠️ **Common confusion — two easy slips in the Beta formula.** The normaliser is $B(\alpha_S,\alpha_F) =
> \Gamma(\alpha_S)\Gamma(\alpha_F)/\Gamma(\alpha_S + \alpha_F)$ — the *product* of Gammas on top, not a single "$\Gamma(\alpha_S, \alpha_F)$"
> divided by them — and *both* powers carry $-1$: $(1-x)^{\alpha_F - 1}$, not $(1-x)^{\alpha_F}$. The mode is
> $(\alpha_S - 1)/\big((\alpha_S - 1) + (\alpha_F - 1)\big)$, with brackets.

In [ ]:
# ── 5.2 The same belief held with more or less conviction: Beta priors with mean 0.3 ──
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for strength, col in zip([2, 10, 50, 200], ["tab:gray", "tab:green", "tab:orange", "tab:red"]):
    a_, b_ = 0.3 * strength, 0.7 * strength
    ax.plot(xg, stats.beta.pdf(xg, a_, b_), color=col, label=f"Beta({a_:g}, {b_:g}),  α + β = {strength}")
    lo, hi = stats.beta.ppf([0.025, 0.975], a_, b_)
    print(f"Beta({a_:4g}, {b_:5g}): mean {a_ / (a_ + b_):.2f}, SD {np.sqrt(a_ * b_ / ((a_ + b_) ** 2 * (a_ + b_ + 1))):.3f}, "
          f"95 % of the prior mass in [{lo:.3f}, {hi:.3f}]")
ax.set_xlabel("θ"); ax.set_ylabel("density"); ax.set_ylim(0, 14); ax.legend(fontsize=7)
ax.set_title("Same centre, growing conviction: α + β acts like a number of imaginary observations")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All four priors have mean 0.3, but their spread differs enormously: with $\alpha + \beta = 2$ the prior
admits almost anything (95 % of its mass in [0.001, 0.891]), with $\alpha + \beta = 200$ it confines $\theta$ to [0.239, 0.365]. Since the
variance is $\alpha\beta/((\alpha+\beta)^2(\alpha+\beta+1))$, the SD shrinks roughly like $1/\sqrt{\alpha + \beta}$ — exactly like the
standard error of a proportion from $\alpha + \beta$ observations (Module 22). Choosing a Beta prior means choosing a centre *and*
how many observations' worth of conviction you have.

✏️ **Your turn.** (a) By hand: mean and mode of Beta(3, 7) and of Beta(30, 70). Which one expresses more certainty, and how can
you tell from the variance formula? (b) Plot Beta(0.5, 0.5). What kind of belief does a U-shaped prior express?

---
## 6 · The Beta posterior: conjugacy

> 📍 **Where we are.** Prior (Section 5) + likelihood (Module 23) → posterior, by Bayes' theorem. For a Beta prior and coin-flip
> data the posterior turns out to be another Beta distribution.

Model the prior as a Beta distribution with hyperparameters $\alpha_{H_0}$ and $\alpha_{T_0}$, i.e.
$p(\theta) = \text{Beta}(\alpha_{H_0}, \alpha_{T_0})$, and observe $\alpha_H$ heads and $\alpha_T$ tails (so $n = \alpha_H + \alpha_T$).
The posterior:

$$\begin{aligned}
p(\theta\mid\mathcal D) &= \frac{p(\mathcal D\mid\theta)\,p(\theta)}{p(\mathcal D)} \;\propto\; p(\mathcal D\mid\theta)\,p(\theta)\\
&= \binom{\alpha_H + \alpha_T}{\alpha_H}\theta^{\alpha_H}(1-\theta)^{\alpha_T}\cdot\frac{\theta^{\alpha_{H_0}-1}(1-\theta)^{\alpha_{T_0}-1}}{B(\alpha_{H_0}, \alpha_{T_0})}\\
&\propto \theta^{\alpha_H + \alpha_{H_0} - 1}(1-\theta)^{\alpha_T + \alpha_{T_0} - 1}
\quad\Longrightarrow\quad \theta\mid\mathcal D \sim \text{Beta}(\alpha_H + \alpha_{H_0},\ \alpha_T + \alpha_{T_0}) .
\end{aligned}$$

*In words:* line 1 drops $p(\mathcal D)$, which does not depend on $\theta$ ("∝" means "equal up to a constant factor"); line 2
writes out the binomial likelihood and the Beta density; line 3 drops the constants $\binom{n}{\alpha_H}$ and $1/B$ and adds the
exponents of $\theta$ and of $(1-\theta)$. What is left has the shape of a Beta density, and since a density is determined by its
shape (only one constant makes it integrate to 1), the posterior *is* that Beta distribution. **The prior's hyperparameters
simply add to the observed counts.** Because prior and posterior belong to the same family, the Beta is called **conjugate** to
the Bernoulli/binomial likelihood.

The same statement in the notation most textbooks use: with prior $\Theta\sim\text{Beta}(\alpha, \beta)$ and $k = \sum_i x_i$
successes in $n$ Bernoulli trials, the posterior is $\text{Beta}(k + \alpha,\ n - k + \beta)$.

**Three summaries of a Beta($a', b'$) posterior** (with $a' = k + \alpha$, $b' = n - k + \beta$):

| summary | formula | meaning |
|---|---|---|
| posterior mean | $\dfrac{k + \alpha}{n + \alpha + \beta}$ | the pseudocount estimate with $c_{H_0} = \alpha$, $c_{T_0} = \beta$ |
| posterior mode (**MAP**) | $\dfrac{k + \alpha - 1}{n + \alpha + \beta - 2}$ | the pseudocount estimate with $c_{H_0} = \alpha - 1$, $c_{T_0} = \beta - 1$ |
| 95 % credible interval | 2.5 % and 97.5 % quantiles of Beta($a', b'$) | $P(\text{lower} \le \theta \le \text{upper}\mid\mathcal D) = 0.95$ |

*In words:* the pseudocount formula of Section 4 is a posterior mean (or, with pseudocounts one smaller, a MAP). A Beta(1, 1)
(flat) prior adds 1 imaginary head and 1 tail to the posterior *mean* but nothing to the MAP.

In [ ]:
# ── 6.1 Prior → posterior for a resistance frequency (SIMULATED surveillance: 3 resistant isolates out of 20) ──
k_res, n_iso = 3, 20
priors = {"flat Beta(1, 1)": (1, 1), "weak Beta(2, 8) (mean 0.2)": (2, 8), "strong, wrong Beta(40, 40) (mean 0.5)": (40, 40)}
th = np.linspace(0.0005, 0.9995, 2000)
lik = th ** k_res * (1 - th) ** (n_iso - k_res)
lik_density = lik / (lik.sum() * (th[1] - th[0]))                      # likelihood rescaled to area 1, for display
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, (name, (a0, b0)) in zip(axes, priors.items()):
    a_post, b_post = k_res + a0, n_iso - k_res + b0
    ax.plot(th, stats.beta.pdf(th, a0, b0), color="tab:gray", ls="--", label="prior")
    ax.plot(th, lik_density, color="tab:orange", label="likelihood (rescaled)")
    ax.plot(th, stats.beta.pdf(th, a_post, b_post), color="tab:blue", lw=2, label=f"posterior Beta({a_post}, {b_post})")
    ax.set_title(name, fontsize=9); ax.set_xlabel("resistance frequency θ"); ax.set_xlim(0, 1)
    lo, hi = stats.beta.ppf([0.025, 0.975], a_post, b_post)
    mode = (a_post - 1) / (a_post + b_post - 2)
    print(f"{name:38s}: posterior Beta({a_post}, {b_post}); mean {a_post / (a_post + b_post):.3f}, MAP {mode:.3f}, "
          f"95 % credible interval [{lo:.3f}, {hi:.3f}]")
axes[0].set_ylabel("density"); axes[0].legend(fontsize=7)
fig.suptitle(f"SIMULATED: {k_res} resistant isolates out of {n_iso} (MLE {k_res / n_iso:.2f})", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 3 resistant isolates out of 20 the MLE is 0.15. Under a flat prior the posterior is
Beta(4, 18): its MAP equals the MLE (0.150), its mean is a little higher (0.182), and the 95 % credible interval is
[0.054, 0.363]. A weak prior centred at 0.2 (worth 10 imaginary isolates) moves the posterior only slightly. A strong prior
centred at 0.5 (worth 80 imaginary isolates) overwhelms 20 real ones: the posterior Beta(43, 57) sits near 0.43,
far from the data. (In the flat panel the posterior and the rescaled likelihood coincide — a flat prior changes nothing.) In every panel the
posterior lies *between* the prior and the likelihood, closer to whichever is
sharper.

**Credible vs confidence intervals.** The credible interval supports the statement Module 22 forbade for confidence
intervals: "given the data (and the prior), there is a 95 % probability that $\theta$ lies in this interval". The price is that
the statement depends on the prior.

In [ ]:
# ── 6.2 Conjugacy in action: updating one isolate at a time gives the same posterior as all at once ──
rng_seq = np.random.default_rng(SEED + 62)
isolates = (rng_seq.random(20) < 0.15).astype(int)
a_seq, b_seq = 1, 1                                   # start from the flat prior
for x_i in isolates:
    a_seq, b_seq = a_seq + x_i, b_seq + (1 - x_i)       # each isolate adds 1 to α (resistant) or to β (susceptible)
print("isolates (1 = resistant):", isolates.tolist())
print(f"one at a time → Beta({a_seq}, {b_seq});  all at once → Beta({1 + isolates.sum()}, {1 + 20 - isolates.sum()})")

🔍 **What just happened?** Starting from Beta(1, 1) and adding one isolate at a time — +1 to the first parameter for each
resistant isolate, +1 to the second for each susceptible one — ends at Beta(3, 19), exactly the posterior obtained from all
20 isolates at once. Today's posterior is tomorrow's prior; for conjugate models updating is just bookkeeping of counts.

In [ ]:
# ── 6.3 The posterior sharpens as isolates accumulate (SIMULATED, true frequency 0.15, flat prior) ──
rng_sh = np.random.default_rng(SEED + 63)
stream = (rng_sh.random(500) < 0.15).astype(int)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for n_, col in zip([0, 5, 20, 100, 500], ["tab:gray", "tab:purple", "tab:green", "tab:orange", "tab:red"]):
    k_ = stream[:n_].sum()
    ax.plot(th, stats.beta.pdf(th, 1 + k_, 1 + n_ - k_), color=col, label=f"n = {n_} ({k_} resistant)")
    lo, hi = stats.beta.ppf([0.025, 0.975], 1 + k_, 1 + n_ - k_)
    print(f"n = {n_:3d}, k = {k_:3d}: posterior Beta({1 + k_}, {1 + n_ - k_}), 95 % credible interval "
          f"[{lo:.3f}, {hi:.3f}] (width {hi - lo:.3f})")
ax.axvline(0.15, color="k", ls=":", lw=1, label="truth 0.15"); ax.set_xlim(0, 0.6)
ax.set_xlabel("resistance frequency θ"); ax.set_ylabel("posterior density"); ax.legend(fontsize=7)
ax.set_title("SIMULATED: more isolates, sharper posterior (flat prior)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Before any data the flat prior spreads its mass evenly (95 % interval [0.025, 0.975]). After 5, 20, 100
and 500 isolates the posterior narrows — credible-interval widths 0.455, 0.226, 0.136 and 0.060 — and concentrates around the true 0.15. The width
shrinks roughly like $1/\sqrt n$, the same law as the standard error of Module 22: Bayesian and frequentist uncertainty agree
when the data dominate the prior.

✏️ **Your turn.** (a) By hand: prior Beta(3, 3), data 4 heads and 3 tails. Give the posterior, its mean and its MAP.
(b) With the flat prior, how many resistant isolates out of 20 make the 95 % credible interval exclude 0.5?

### 📘 Lecture notes — pseudocounts, the Beta distribution and the Beta posterior

**Pseudocounts.** $\hat\alpha_{\text{MLE}} = \frac{c_H}{c_H + c_T}$ assumes we know nothing ahead of time. Adding imaginary counts
$c_{H_0}, c_{T_0}$: $\hat\alpha_{\text{MLE-PC}} = \frac{c_H + c_{H_0}}{c_H + c_T + c_{H_0} + c_{T_0}}$. *Example:* $c_H = 0$,
$c_T = 100$: MLE $0$; $c_{H_0} = c_{T_0} = 1$ → $1/102$; $c_{H_0} = c_{T_0} = 100$ → $100/300 = 1/3$. Larger pseudocounts = stronger
prior belief = greater effect; as $n\to\infty$ the data dominate.

**Beta distribution.** $f(x;\alpha_S,\alpha_F) = \frac{x^{\alpha_S-1}(1-x)^{\alpha_F-1}}{B(\alpha_S,\alpha_F)}$,
$B(\alpha_S,\alpha_F) = \frac{\Gamma(\alpha_S)\Gamma(\alpha_F)}{\Gamma(\alpha_S+\alpha_F)}$, $\Gamma(a) = (a-1)!$. Mean
$\frac{\alpha_S}{\alpha_S+\alpha_F}$; mode $\frac{\alpha_S-1}{\alpha_S+\alpha_F-2}$ ($\alpha_S,\alpha_F > 1$). Shapes: (1,1) flat, (2,2)
hill, (2,6) and (6,2) skewed.

**Conjugacy.** Prior $p(\theta) = \text{Beta}(\alpha_{H_0}, \alpha_{T_0})$, data $\alpha_H$ heads and $\alpha_T$ tails:

$$p(\theta\mid\mathcal D)\propto\theta^{\alpha_H}(1-\theta)^{\alpha_T}\cdot\theta^{\alpha_{H_0}-1}(1-\theta)^{\alpha_{T_0}-1}
\;\Rightarrow\; \text{Beta}(\alpha_H + \alpha_{H_0},\ \alpha_T + \alpha_{T_0}) .$$

*In words:* hyperparameters add to counts. Equivalently, prior Beta($\alpha, \beta$) and $k$ successes in $n$ → Beta($k+\alpha$,
$n-k+\beta$). Posterior mean $\frac{k+\alpha}{n+\alpha+\beta}$ = pseudocount estimate; MAP $\frac{k+\alpha-1}{n+\alpha+\beta-2}$.

**Common confusions.** It is the **prior** that we model as a Beta (and the **posterior** that comes out as a Beta). A credible
interval is a probability statement about $\theta$ given the data; a confidence interval is a statement about the procedure.

> 🎓 **Exam tip.** To derive a conjugate posterior: multiply likelihood × prior, drop every factor without $\theta$, add the
> exponents, and *recognise* the family. Do not compute $p(\mathcal D)$ — you never need it.

---
## 7 · Maximum a posteriori (MAP) estimation

> 📍 **Where we are.** The posterior is a whole distribution. Often we still want one number — and the most natural one is
> the posterior's peak.

**Definition.** As a reminder, $\hat\theta_{\text{MLE}} = \arg\max_{\theta\in\Theta} p(\mathcal D\mid\theta)$. If we want to include
information from our prior knowledge, we use the **maximum a posteriori** (MAP) estimate:

$$\hat\theta_{\text{MAP}} = \operatorname*{arg\,max}_{\theta\in\Theta}\ p(\theta\mid\mathcal D) = \operatorname*{arg\,max}_{\theta\in\Theta}\ p(\mathcal D\mid\theta)\cdot p(\theta) .$$

*In words:* choose the parameter value with the highest posterior probability (density). The second form drops $p(\mathcal D)$,
which does not depend on $\theta$ and so cannot move the maximum. A flat prior makes MAP and MLE coincide.

Note that in both cases we are making a **point estimate** of $\theta$. We still do not have the whole picture — a single number
hides the posterior's spread — so MAP is only **partially Bayesian**. The fully Bayesian answer is the posterior distribution
itself, summarised by its mean, its mode (the MAP) and a credible interval.

### 7.1 MAP with an unknown prior: the Bernoulli with a Beta prior

When the prior is not known we choose one, then follow the same procedure as for the MLE: take the log and find the zero of
its derivative,

$$\frac{d}{d\theta}\ln\big[p(\mathcal D\mid\theta)\,p(\theta)\big] = 0 .$$

*In words:* the log of a product is a sum, $\ln p(\mathcal D\mid\theta) + \ln p(\theta)$, so the MAP equation is the MLE equation
plus the derivative of the log-prior.

Suppose we want the probability of heads $p$ of a new coin (1 = H, 0 = T), $X_1,\dots,X_n\sim\text{Bernoulli}(p)$, with
$\alpha_H$ heads and $\alpha_T$ tails, and a $\text{Beta}(\alpha_{H_0}, \alpha_{T_0})$ prior. Using $\log(MN) = \log M + \log N$
and $\log M^k = k\log M$ (constants dropped, since they do not move the arg max):

$$\begin{aligned}
\hat\theta_{\text{MAP}} &= \operatorname*{arg\,max}_{p}\ \log P(p\mid\mathcal D) = \operatorname*{arg\,max}_{p}\ \big[\log P(\mathcal D\mid p) + \log P(p)\big]\\
&= \operatorname*{arg\,max}_{p}\ \big[\log p^{\alpha_H}(1-p)^{\alpha_T} + \log p^{\alpha_{H_0}-1}(1-p)^{\alpha_{T_0}-1}\big]\\
&= \operatorname*{arg\,max}_{p}\ \big[\alpha_H\log p + (\alpha_{H_0}-1)\log p + \alpha_T\log(1-p) + (\alpha_{T_0}-1)\log(1-p)\big]\\
&= \operatorname*{arg\,max}_{p}\ \big[(\alpha_H + \alpha_{H_0} - 1)\log p + (\alpha_T + \alpha_{T_0} - 1)\log(1-p)\big] .
\end{aligned}$$

*In words:* the log-posterior looks exactly like a Bernoulli log-likelihood with $\alpha_H + \alpha_{H_0} - 1$ "heads" and
$\alpha_T + \alpha_{T_0} - 1$ "tails". Differentiate with respect to $p$ (the derivative of $\log(1-p)$ brings a factor $-1$):

$$\frac{\alpha_H + \alpha_{H_0} - 1}{p} - \frac{\alpha_T + \alpha_{T_0} - 1}{1-p} = 0
\;\Longrightarrow\; (1-p)(\alpha_H + \alpha_{H_0} - 1) = p\,(\alpha_T + \alpha_{T_0} - 1) .$$

Expand both sides and collect the $p$ terms on the right:

$$\alpha_H + \alpha_{H_0} - 1 - p(\alpha_H + \alpha_{H_0} - 1) = p(\alpha_T + \alpha_{T_0} - 1)
\;\Longrightarrow\; \alpha_H + \alpha_{H_0} - 1 = p\,(\alpha_H + \alpha_{H_0} + \alpha_T + \alpha_{T_0} - 2) ,$$

$$\hat\theta_{\text{MAP}} = \frac{\alpha_H + \alpha_{H_0} - 1}{\alpha_H + \alpha_T + \alpha_{H_0} + \alpha_{T_0} - 2} = \frac{k + (\alpha - 1)}{n + (\alpha - 1) + (\beta - 1)} .$$

*In words:* the MAP is the fraction of heads after adding $\alpha_{H_0} - 1$ imaginary heads and $\alpha_{T_0} - 1$ imaginary tails —
exactly the mode of the Beta posterior of Section 6. (The last form uses the notation prior Beta($\alpha, \beta$), $k$ heads in
$n$ flips.) All four counts enter the denominator with a **plus** sign.

In [ ]:
# ── 7.1 SymPy check of the Beta-prior MAP, and a numerical example ──
pp, aH, aT, aH0, aT0 = sp.symbols("p alpha_H alpha_T alpha_H0 alpha_T0", positive=True)
log_post = (aH + aH0 - 1) * sp.log(pp) + (aT + aT0 - 1) * sp.log(1 - pp)
map_sym = sp.solve(sp.diff(log_post, pp), pp)[0]
print("MAP =", sp.simplify(map_sym))
example = {aH: 6, aT: 4, aH0: 3, aT0: 3}
print("6 heads, 4 tails, prior Beta(3, 3): MAP =", map_sym.subs(example), "=", float(map_sym.subs(example)),
      "| MLE = 0.6 | posterior mean = (6 + 3)/(10 + 6) =", 9 / 16)
grid_p = np.linspace(0.001, 0.999, 9981)
lp = 8 * np.log(grid_p) + 6 * np.log(1 - grid_p)
print("grid maximum of the log-posterior:", grid_p[lp.argmax()].round(4))

🔍 **What just happened?** SymPy solves the derivative to $(\alpha_H + \alpha_{H_0} - 1)/(\alpha_H + \alpha_T + \alpha_{H_0} + \alpha_{T_0} - 2)$
— every count with a plus sign. For 6 heads and 4 tails with a Beta(3, 3) prior the MAP is $8/14 = 4/7 \approx 0.571$, between
the MLE 0.6 and the prior's centre 0.5, and slightly different from the posterior mean $9/16 = 0.5625$ (mode and mean of a
skewed Beta differ). A grid search over the log-posterior agrees.

In [ ]:
# ── 7.2 Three point estimates on one posterior: 1 resistant isolate out of 6, prior Beta(2, 2) ──
k1, n1, a0, b0 = 1, 6, 2, 2
a_p, b_p = k1 + a0, n1 - k1 + b0
point_estimates = {"MLE k/n": k1 / n1, "MAP (posterior mode)": (a_p - 1) / (a_p + b_p - 2),
                   "posterior median": stats.beta.median(a_p, b_p), "posterior mean": a_p / (a_p + b_p)}
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(th, stats.beta.pdf(th, a_p, b_p), color="tab:blue", lw=2, label=f"posterior Beta({a_p}, {b_p})")
for (name, v), col in zip(point_estimates.items(), ["tab:orange", "tab:red", "tab:purple", "tab:green"]):
    ax.axvline(v, color=col, ls="--", label=f"{name} = {v:.3f}")
    print(f"{name:21s} = {v:.4f}")
ax.set_xlim(0, 1); ax.set_xlabel("resistance frequency θ"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("A skewed posterior: its mode, median and mean differ")
plt.tight_layout(); plt.show()

🔍 **What just happened?** One resistant isolate out of six gives MLE $1/6 = 0.167$. With a Beta(2, 2) prior the posterior is
Beta(3, 7), which is right-skewed: its peak (the MAP) is $2/8 = 0.250$, its median 0.286 and its mean $3/10 = 0.300$. The three
Bayesian summaries differ because the posterior is not symmetric; all three are pulled from the MLE toward the prior's centre
0.5. Which to report is a choice: the MAP is the single most plausible value, the mean minimises the expected squared error,
and the posterior itself (with a credible interval) is the complete answer.

### 7.3 A discrete prior

Samples $x = (0, 0, 1, 1, 0)$ come from a Bernoulli($\theta$), and we restrict $\theta\in\{0.2, 0.5, 0.7\}$ with the (discrete)
prior $p(0.2) = 0.1$, $p(0.5) = 0.01$, $p(0.7) = 0.89$. What is the MAP for $\theta$? First the likelihood,
$L(x\mid\theta) = \theta^2(1-\theta)^3$; then multiply by the prior **of the same $\theta$**.

In [ ]:
# ── 7.3 The discrete-prior example: likelihood, Bayes numerator, posterior ──
x_small = np.array([0, 0, 1, 1, 0])
thetas = np.array([0.2, 0.5, 0.7])
prior_disc = np.array([0.1, 0.01, 0.89])
lik_disc = thetas ** x_small.sum() * (1 - thetas) ** (len(x_small) - x_small.sum())
table = bayes_table([f"θ = {t}" for t in thetas], prior_disc, lik_disc)
for t, L_, pr in zip(thetas, lik_disc, prior_disc):
    print(f"L(x | {t}) = {t}² · {1 - t:.1f}³ = {L_:.5f};   × p({t}) = {pr} → {L_ * pr:.7f}")
print(); print(table.to_string(float_format=lambda v: f"{v:.5g}"))
print(f"\nMAP = {thetas[table['posterior p(θ|D)'].argmax()]} | MLE over the three values = {thetas[lik_disc.argmax()]} | "
      f"MLE over [0, 1] = {x_small.mean()}")

🔍 **What just happened?** The likelihoods are $0.2^2\cdot0.8^3 = 0.02048$, $0.5^2\cdot0.5^3 = 0.03125$ and $0.7^2\cdot0.3^3 = 0.01323$:
the data favour $\theta = 0.5$ (the MLE among the three values; the unrestricted MLE would be $2/5 = 0.4$). Multiplying each by
its own prior gives numerators $0.002048$, $0.0003125$ and $0.01323\times0.89 = 0.0117747$, and normalising gives posteriors
0.145, 0.022 and 0.833. The prior's 0.89 on $\theta = 0.7$ outweighs the data: $\hat\theta_{\text{MAP}} = 0.7$.

> ⚠️ **Check the pairing.** Each likelihood must be multiplied by the prior *of the same hypothesis*:
> $p(0.7\mid x)\propto L(x\mid 0.7)\,p(0.7) = 0.7^2\cdot0.3^3\cdot0.89 = 0.0117747$. Writing $0.2^2\cdot0.8^3\cdot0.89$ or
> $p(0.5)$ in that line would be a different (wrong) product, even if the MAP happened to come out the same. Note also that the
> three numerators are *unnormalised*; the posterior probabilities are the numerators divided by their sum, 0.0141352.

### 7.4 MLE, MAP or posterior mean? A simulation

Which estimate is closest to the truth on average? We simulate 200,000 surveillance studies of $n$ isolates with a true
resistance frequency of 0.3 and compare the **mean squared error** (MSE, average squared distance to the truth) of the MLE, the
MAP and the posterior mean under four priors: flat Beta(1, 1), weak Beta(2, 2), a sensible informative Beta(3, 7) (centred on
the truth) and a confident wrong Beta(14, 6) (centred at 0.7).

In [ ]:
# ── 7.4 Mean squared error of MLE, MAP and posterior mean (SIMULATED studies, true resistance frequency 0.3) ──
rng_mse = np.random.default_rng(SEED + 73)
rows = []
for n_ in [5, 20, 100]:
    k_ = rng_mse.binomial(n_, 0.3, 200_000)
    row = {"n": n_, "MLE k/n": np.mean((k_ / n_ - 0.3) ** 2)}
    for name, (a0, b0) in {"Beta(1,1)": (1, 1), "Beta(2,2)": (2, 2), "Beta(3,7)": (3, 7), "Beta(14,6) wrong": (14, 6)}.items():
        row[f"MAP {name}"] = np.mean(((k_ + a0 - 1) / (n_ + a0 + b0 - 2) - 0.3) ** 2)
        row[f"mean {name}"] = np.mean(((k_ + a0) / (n_ + a0 + b0) - 0.3) ** 2)
    rows.append(row)
mse = pd.DataFrame(rows).set_index("n").T
print("mean squared error (rows: estimator; columns: number of isolates n):")
print(mse.to_string(float_format=lambda v: f"{v:.4f}"))

🔍 **What just happened?** With $n = 5$ the MLE's MSE is 0.0419; the MAP under a flat prior is identical (flat prior ⇒ MAP =
MLE), but the posterior mean under the same flat prior already does better (0.0246) because it never jumps to 0 or 1.
The sensible prior Beta(3, 7) cuts the error to 0.0072 (MAP) — about six-fold better than the MLE. The confident
*wrong* prior Beta(14, 6) is the worst estimator at $n = 5$ (0.111) and still hurts at $n = 20$. By $n = 100$ the flat and
weak priors are indistinguishable from the MLE (MSE 0.0020–0.0021), the sensible prior still helps a little (0.0017–0.0018), and
the wrong prior still costs (0.0057–0.0059): a prior worth 20 imaginary isolates takes a lot of data to overcome. **A prior is a bet: it pays off with few data when it is sensible,
costs you when it is confidently wrong, and stops mattering as data accumulate.**

### 🌉 Bridge — rare variants and "zero observed"

A variant observed in **0 of 100** chromosomes has MLE frequency 0. With a flat Beta(1, 1) prior the posterior is Beta(1, 101):
its MAP is still 0 (flat prior ⇒ MAP = MLE), but its mean is $1/102$ and its 95 % upper credible bound is
$1 - 0.05^{1/101} \approx 0.029$ — close to the classic "rule of three" ($3/n$) for zero events. Large reference databases of human
genetic variation report allele frequencies with exactly this kind of uncertainty in mind, and Bayesian genotype callers use
priors on allele frequencies to decide whether a few odd reads are a real variant or a sequencing error
([BiologyToML 35 · statistical genetics](../../../../ml/tutorial/BiologyToML/notebooks/35_statistical_genetics.ipynb)).

In [ ]:
# ── 7.5 Zero copies of a variant in 100 chromosomes: MLE, MAP, posterior mean and upper bound (flat prior) ──
k0, n0 = 0, 100
a_post, b_post = 1 + k0, 1 + n0 - k0
print(f"posterior Beta({a_post}, {b_post}): MLE = {k0 / n0}, MAP = {(a_post - 1) / (a_post + b_post - 2)}, "
      f"posterior mean = {a_post / (a_post + b_post):.4f} (= 1/102)")
print(f"95 % upper credible bound = {stats.beta.ppf(0.95, a_post, b_post):.4f}  (formula 1 − 0.05^(1/101) = "
      f"{1 - 0.05 ** (1 / 101):.4f}; rule of three 3/n = {3 / n0:.3f})")

🔍 **What just happened?** For 0 copies in 100 chromosomes the MLE and the flat-prior MAP are both 0, the posterior mean is
0.0098, and we can be 95 % sure (given the flat prior) that the frequency is below 0.029. A point estimate of 0 hides that
range; the posterior shows it.

✏️ **Your turn.** (a) By hand: 2 heads in 8 flips, prior Beta(2, 2) — MLE, MAP and posterior mean. (b) In 7.4 add the
prior Beta(30, 70) (centred on the truth, worth 100 isolates). At which $n$ does it stop beating the MLE by a factor of 2?

---
## 8 · Conjugate priors, and the Poisson with a Gamma prior

> 📍 **Where we are.** The Beta was conjugate to the Bernoulli. Every exponential-family likelihood has a conjugate prior
> (Section 1 is why); here is the general idea and the count-data case.

**Definition.** If the prior and the posterior belong to the same parametric family, the prior is said to be **conjugate** for the
likelihood. The parameters of the prior distribution are called **hyperparameters** (e.g. $\alpha_S, \alpha_F$ in
Beta($\alpha_S, \alpha_F$)) — to distinguish them from the parameter $\theta$ itself.

**How do you set the hyperparameters?** From prior knowledge (experienced domain-specific experts, previous studies,
databases), or, with no prior knowledge, with an **"uninformative"** prior such as Beta(1, 1) (flat). A third option, used
throughout genomics, is to estimate them from many related data sets at once (**empirical Bayes**, Section 8.3).

| Prior | Likelihood | Posterior | update (rate form for the Gamma) |
|---|---|---|---|
| Beta($\alpha, \beta$) | Bernoulli($\theta$), $k$ successes in $n$ | Beta | Beta($\alpha + k$, $\beta + n - k$) |
| Beta($\alpha, \beta$) | Binomial($m, \theta$), $k$ successes | Beta | Beta($\alpha + k$, $\beta + m - k$) |
| Gamma($\alpha, \beta$) | Poisson($\lambda$), counts $x_1..x_n$ | Gamma | Gamma($\alpha + \sum x_i$, $\beta + n$) |
| Normal($\mu_0, \sigma_0^2$) | Normal($\mu, \sigma^2$), $\sigma^2$ known | Normal | Section 9 |

**The Gamma distribution.** Counts have a rate $\lambda > 0$, so the prior must live on $(0, \infty)$. The conjugate prior of the
Poisson is the **Gamma** distribution. In *shape–scale* form (shape $k$, scale $\theta$):

$$f(\lambda) = \frac{1}{\Gamma(k)\,\theta^k}\,\lambda^{k-1}e^{-\lambda/\theta}, \qquad \lambda > 0 ;$$

*in words:* a power of $\lambda$ times a decaying exponential; mean $k\theta$, mode $(k-1)\theta$ for $k \ge 1$. In *shape–rate*
form, with rate $\beta = 1/\theta$ and shape $\alpha = k$: $f(\lambda) = \frac{\beta^\alpha}{\Gamma(\alpha)}\lambda^{\alpha-1}e^{-\beta\lambda}$,
mean $\alpha/\beta$. (`scipy.stats.gamma(a=k, scale=θ)` uses the shape–scale form.)

> ⚠️ **Common confusion.** The exponent is $-\lambda/\theta$ (the variable divided by the scale), not $-\alpha/\theta$; and do not
> reuse the letter $k$ for a Poisson count and for the Gamma shape in the same calculation, nor $\theta$ for both "the
> parameter" and "the Gamma scale". Below: counts $x_i$, shape $k$, scale $\theta$, rate $\beta = 1/\theta$.

**MAP for the Poisson with a Gamma prior. Step 1 — the log of likelihood × prior.**

$$P(\mathcal D\mid\lambda)P(\lambda) = e^{-n\lambda}\frac{\lambda^{\sum_i x_i}}{\prod_i x_i!}\cdot\frac{\lambda^{k-1}e^{-\lambda/\theta}}{\theta^k\,\Gamma(k)},$$

$$\ln\big(P(\mathcal D\mid\lambda)P(\lambda)\big) = \ln\lambda\Big(k - 1 + \sum_{i=1}^n x_i\Big) - \lambda\Big(n + \frac1\theta\Big) - \sum_{i=1}^n\ln(x_i!) - k\ln\theta - \ln\Gamma(k) .$$

*In words:* collect all powers of $\lambda$ (from the likelihood, $\sum x_i$; from the prior, $k - 1$) and all terms linear in
$\lambda$ (from the likelihood, $-n\lambda$; from the prior, $-\lambda/\theta$); the last three terms do not involve $\lambda$.

**Step 2 — maximise.** The last three terms have derivative 0:

$$\frac{d}{d\lambda}: \quad \frac1\lambda\Big(k - 1 + \sum_{i=1}^n x_i\Big) - \Big(n + \frac1\theta\Big) = 0
\;\Longrightarrow\; \hat\lambda_{\text{MAP}} = \frac{k - 1 + \sum_{i=1}^n x_i}{n + 1/\theta} = \frac{\alpha - 1 + \sum_i x_i}{\beta + n} .$$

*In words:* the Gamma prior acts like $k - 1$ extra counts observed in $1/\theta = \beta$ extra cells. And the posterior itself:
$P(\lambda\mid\mathcal D)\propto\lambda^{(k + \sum x_i) - 1}e^{-\lambda(n + 1/\theta)}$ — a Gamma with shape $k + \sum_i x_i$ and rate
$n + 1/\theta$, i.e. **Gamma($\alpha + \sum_i x_i$, $\beta + n$)** in rate form; posterior mean $(\alpha + \sum x_i)/(\beta + n)$.

In [ ]:
# ── 8.1 Gamma–Poisson: a gene in a rare cell type, measured in only 3 cells (SIMULATED, true rate 2.4) ──
rng_gp = np.random.default_rng(SEED + 81)
x_cells = rng_gp.poisson(2.4, 3)
shape_k, scale_theta = 4.0, 0.5                          # prior: mean kθ = 2 molecules per cell, rate β = 1/θ = 2
a_post, rate_post = shape_k + x_cells.sum(), 1 / scale_theta + len(x_cells)
lam_map = (shape_k - 1 + x_cells.sum()) / (len(x_cells) + 1 / scale_theta)
print("counts in 3 cells:", x_cells.tolist(), f"| MLE x̄ = {x_cells.mean():.3f}")
print(f"prior Gamma(shape {shape_k}, scale {scale_theta}) → posterior Gamma(shape {a_post}, rate {rate_post}) "
      f"[scale {1 / rate_post:.3f}]")
print(f"MAP = (k − 1 + Σx)/(n + 1/θ) = ({shape_k:.0f} − 1 + {x_cells.sum()})/({len(x_cells)} + {1 / scale_theta:.0f}) = {lam_map:.3f}"
      f" | posterior mean = {a_post / rate_post:.3f}")
lo, hi = stats.gamma.ppf([0.025, 0.975], a=a_post, scale=1 / rate_post)
print(f"95 % credible interval [{lo:.2f}, {hi:.2f}]   🧪 true rate 2.4")
lg = np.linspace(0.01, 8, 800)
post = stats.gamma.pdf(lg, a=a_post, scale=1 / rate_post)
print("grid maximum of the posterior density:", lg[post.argmax()].round(3))

🔍 **What just happened?** Three cells contain 2, 3 and 1 molecules (MLE 2.0). The prior Gamma(shape 4, scale 0.5) says
"about 2 molecules per cell, give or take" and is worth $1/\theta = 2$ imaginary cells. The posterior is Gamma(shape 10,
rate 5); its peak, $\hat\lambda_{\text{MAP}} =$ 1.8, matches the formula and a grid search, and the posterior mean is 2.0.
The 95 % credible interval [0.96, 3.42] honestly reports how little three cells reveal (it contains the true 2.4).

In [ ]:
# ── 8.2 Prior, likelihood and posterior for the three cells ──
lik_g = stats.poisson.pmf(x_cells[:, None], lg[None, :]).prod(axis=0)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(lg, stats.gamma.pdf(lg, a=shape_k, scale=scale_theta), "--", color="tab:gray", label="prior Gamma(shape 4, scale 0.5)")
ax.plot(lg, lik_g / (lik_g.sum() * (lg[1] - lg[0])), color="tab:orange", label="likelihood of the three counts (rescaled)")
ax.plot(lg, post, color="tab:blue", lw=2, label=f"posterior Gamma(shape {a_post:g}, rate {rate_post:g})")
ax.axvline(2.4, color="k", ls=":", lw=1, label="true rate 2.4")
ax.set_xlabel("expression rate λ (molecules per cell)"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("SIMULATED: three cells and a Gamma prior")
plt.tight_layout(); plt.show()
print(f"likelihood peak (MLE) {lg[lik_g.argmax()]:.2f} | prior mode (k − 1)θ = {(shape_k - 1) * scale_theta:.2f} | "
      f"posterior mode {lg[post.argmax()]:.2f}")

🔍 **What just happened?** The prior (grey) peaks at $(k-1)\theta = 1.5$, the likelihood of the three counts (orange) at the
MLE 2.0, and the posterior (blue) at 1.8 — between them, and narrower than either, because it combines the information of the
prior (worth 2 cells) and the data (3 cells). With only three cells the prior pulled the estimate *away* from the true 2.4
here: a prior centred at 2 is a reasonable but not perfect guess, and its influence fades as more cells are measured.

### 8.3 Empirical Bayes: learning the prior from many genes

Where does a prior like Gamma(4, 0.5) come from? In genomics we measure *thousands* of genes at once. Their rates form a
distribution — the prior — and we can estimate its hyperparameters from all genes together, then use it to stabilise each
gene's estimate. This is **empirical Bayes**. Below, 2,000 genes have true rates drawn from a Gamma distribution, each measured
in only 3 cells of a rare cell type (SIMULATED). We estimate the prior by matching moments and compare per-gene MLEs with the
empirical-Bayes posterior means.

In [ ]:
# ── 8.3 Empirical-Bayes shrinkage of 2,000 expression rates measured in 3 cells each (SIMULATED) ──
rng_eb = np.random.default_rng(SEED + 82)
n_genes, n_c = 2000, 3
true_lam = rng_eb.gamma(2.0, 2.0, n_genes)                      # true rates: Gamma(shape 2, scale 2), mean 4
obs = rng_eb.poisson(true_lam[:, None], (n_genes, n_c))
mle = obs.mean(axis=1)
m, v = mle.mean(), mle.var()
var_lam = v - m / n_c                                             # Var(x̄) = Var(λ) + E[λ]/n  ⇒  Var(λ) ≈ v − m/n
alpha_hat, beta_hat = m ** 2 / var_lam, m / var_lam              # Gamma(shape α, rate β) with mean α/β, variance α/β²
post_mean = (alpha_hat + obs.sum(axis=1)) / (beta_hat + n_c)
print(f"estimated prior: Gamma(shape {alpha_hat:.2f}, rate {beta_hat:.3f}) — true Gamma(shape 2, rate 0.5)")
print(f"MSE of per-gene MLE x̄: {np.mean((mle - true_lam) ** 2):.3f};  MSE of empirical-Bayes posterior mean: "
      f"{np.mean((post_mean - true_lam) ** 2):.3f}")
zero = mle == 0
print(f"{zero.sum()} genes have x̄ = 0 (MLE 'not expressed'); their EB estimates are {post_mean[zero].min():.2f}; "
      f"their true rates average {true_lam[zero].mean():.2f}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True, sharey=True)
for ax, est, col, name in [(axes[0], mle, "tab:red", "per-gene MLE (3 cells)"),
                           (axes[1], post_mean, "tab:blue", "empirical-Bayes posterior mean")]:
    ax.scatter(true_lam, est, s=5, alpha=0.3, color=col)
    ax.plot([0, 15], [0, 15], "k--", lw=1)
    ax.set_xlim(0, 15); ax.set_ylim(0, 15); ax.set_xlabel("true rate λ")
    ax.set_title(f"{name}: MSE {np.mean((est - true_lam) ** 2):.2f}")
axes[0].set_ylabel("estimate")
fig.suptitle("SIMULATED: 2,000 genes — shrinkage pulls noisy estimates toward the population of genes", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From the 2,000 noisy gene means alone, matching moments recovers a prior close to the truth
(Gamma(shape 2.10, rate 0.516) against the true Gamma(shape 2, rate 0.5)). Using it, each gene's estimate becomes $(\hat\alpha + \sum x)/(\hat\beta + 3)$ — the gene's own counts plus a few
imaginary cells' worth of "typical gene". The right-hand cloud is tighter than the left-hand one — the wild overestimates of the MLE are gone — at the price of a visible
bias: the genes with the very highest true rates are now pulled down, toward the average gene. On balance the mean squared error
falls from 1.380 to 1.179 (15 % lower). The 34 genes with zero counts in all three cells are no longer declared "silent": they get the small
positive estimate 0.60 — very close to the average of their true rates (0.60).

### 🌉 Bridge — shrinkage in RNA-seq analysis

This is, in miniature, what the standard differential-expression tools do. **edgeR** and **DESeq2** model read counts with a
negative binomial (a Gamma–Poisson mixture — Module 23's over-dispersion) and estimate each gene's dispersion by borrowing
strength across genes: genes with few reads are shrunk toward the trend of all genes (Robinson, McCarthy & Smyth 2010; Love,
Huber & Anders 2014). DESeq2 also shrinks log fold changes with a normal-type prior (the Normal–Normal model of Section 9), so
that a gene with a huge fold change from three noisy reads is not ranked first. **limma** does the same for gene-wise variances
(Smyth 2004). See [BiologyToML 24 · GLMs for count data](../../../../ml/tutorial/BiologyToML/notebooks/24_glm_count_data.ipynb) and
[BiologyToML 21 · Bayesian learning and MCMC](../../../../ml/tutorial/BiologyToML/notebooks/21_bayesian_learning_mcmc.ipynb) for when no conjugate prior exists.

✏️ **Your turn.** (a) By hand: counts 0, 1, 0 and prior Gamma(shape 3, rate 1). Posterior? MAP? Posterior mean?
(b) In 8.3 use 30 cells per gene instead of 3. How much does empirical Bayes still help, and why?

---
## 9 · The normal mean with a normal prior: shrinkage

> 📍 **Where we are.** The last conjugate pair: continuous measurements with a normal likelihood, and a normal prior on the
> mean. The answer is the most important formula of Bayesian statistics in practice — a precision-weighted average.

**Prior:** $\mu\sim\mathcal N(\mu_0, \sigma_0^2)$, $P(\mu) = \frac{1}{\sqrt{2\pi}\sigma_0}e^{-\frac{1}{2\sigma_0^2}(\mu-\mu_0)^2}$.
**Likelihood** of one observation $x$, with $\sigma^2$ known: $P(x\mid\mu) = \frac{1}{\sqrt{2\pi}\sigma}e^{-\frac{1}{2\sigma^2}(x-\mu)^2}$.

**Step 1 — the log of likelihood × prior.** Since $\frac{1}{\sqrt{2\pi}\sigma}\cdot\frac{1}{\sqrt{2\pi}\sigma_0} = \frac{1}{2\pi\sigma\sigma_0}$,

$$\ln\big(P(x\mid\mu)P(\mu)\big) = -\ln 2\pi - \ln\sigma - \ln\sigma_0 - \frac12\Big[\frac{(x-\mu)^2}{\sigma^2} + \frac{(\mu-\mu_0)^2}{\sigma_0^2}\Big] .$$

*In words:* two squared distances — from $\mu$ to the observation and from $\mu$ to the prior centre — each measured in units of
its own variance; the first three terms are constants.

**Step 2 — maximise over $\mu$.** The constants have derivative 0; the chain rule gives $\frac{d}{d\mu}(x-\mu)^2 = 2(x-\mu)(-1)$ and
$\frac{d}{d\mu}(\mu-\mu_0)^2 = 2(\mu-\mu_0)(1)$:

$$-\frac12\Big[2\frac{(x-\mu)}{\sigma^2}(-1) + 2\frac{(\mu-\mu_0)}{\sigma_0^2}(1)\Big] = 0 \;\Longrightarrow\;
\frac{x-\mu}{\sigma^2} - \frac{\mu-\mu_0}{\sigma_0^2} = 0 .$$

Multiply by $\sigma^2\sigma_0^2$ and collect $\mu$: $(x - \mu)\sigma_0^2 = (\mu - \mu_0)\sigma^2 \Rightarrow x\sigma_0^2 + \mu_0\sigma^2 = \mu(\sigma_0^2 + \sigma^2)$, so

$$\mu_{\text{MAP}} = \frac{x\,\sigma_0^2 + \mu_0\,\sigma^2}{\sigma_0^2 + \sigma^2} = \frac{\frac{1}{\sigma^2}\,x + \frac{1}{\sigma_0^2}\,\mu_0}{\frac{1}{\sigma^2} + \frac{1}{\sigma_0^2}} .$$

*In words:* a weighted average of the observation and the prior mean, each weighted by its **precision** (one over its variance).
A precise measurement (small $\sigma^2$) dominates; a vague prior (large $\sigma_0^2$) barely matters.

**$n$ observations.** Replacing the single $(x-\mu)^2/\sigma^2$ by $\sum_i(x_i-\mu)^2/\sigma^2$ and repeating the steps gives

$$\mu_{\text{MAP}} = \frac{\frac{n}{\sigma^2}\,\bar x + \frac{1}{\sigma_0^2}\,\mu_0}{\frac{n}{\sigma^2} + \frac{1}{\sigma_0^2}}, \qquad
\text{posterior: } \mu\mid\mathcal D \sim \mathcal N\Big(\mu_{\text{MAP}},\ \sigma_{\text{post}}^2\Big),\quad \sigma_{\text{post}}^2 = \frac{1}{\frac{n}{\sigma^2} + \frac{1}{\sigma_0^2}} .$$

*In words:* the data contribute precision $n/\sigma^2$ (it grows with $n$), the prior contributes $1/\sigma_0^2$; the posterior
precision is their sum. The log-posterior is a downward parabola in $\mu$ with $\mu^2$-coefficient $-\frac12(n/\sigma^2 + 1/\sigma_0^2)$,
which is why the posterior is normal with that precision. Because a normal posterior is symmetric, its mode (MAP), mean and median
coincide. As $n\to\infty$, $\mu_{\text{MAP}}\to\bar x$ (the MLE).

> ⚠️ **Common confusion.** The quantity $1/(1/\sigma^2 + 1/\sigma_0^2)$ (for one observation) is the **posterior variance of $\mu$**
> — how uncertain we remain about the mean — not a "MAP estimate of $\sigma^2$". In this model $\sigma^2$ is known; nothing is
> being estimated about it. Do not label it $\sigma^2_{\text{MAP}}$.

In [ ]:
# ── 9.1 Shrinking a log2 fold change measured in 3 replicates (SIMULATED; σ = 0.8 per replicate, prior N(0, 0.5²)) ──
sigma_rep, n_rep = 0.8, 3
mu0, sigma0 = 0.0, 0.5                        # prior: most genes change little
xbar_gene = 1.2                               # this gene's observed mean log2 fold change
prec_data, prec_prior = n_rep / sigma_rep ** 2, 1 / sigma0 ** 2
mu_map = (prec_data * xbar_gene + prec_prior * mu0) / (prec_data + prec_prior)
sd_post = np.sqrt(1 / (prec_data + prec_prior))
print(f"precisions: data n/σ² = {prec_data:.3f}, prior 1/σ0² = {prec_prior:.3f}; weight on the data = "
      f"{prec_data / (prec_data + prec_prior):.3f}")
print(f"MLE x̄ = {xbar_gene}, MAP = posterior mean = {mu_map:.3f}, posterior SD = {sd_post:.3f}")
mus = np.linspace(-1.5, 3, 600)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(mus, stats.norm.pdf(mus, mu0, sigma0), "--", color="tab:gray", label="prior N(0, 0.5²)")
ax.plot(mus, stats.norm.pdf(mus, xbar_gene, sigma_rep / np.sqrt(n_rep)), color="tab:orange",
        label=f"likelihood (∝ N(x̄ = {xbar_gene}, σ²/n))")
ax.plot(mus, stats.norm.pdf(mus, mu_map, sd_post), color="tab:blue", lw=2,
        label=f"posterior N({mu_map:.2f}, {sd_post:.2f}²)")
ax.set_xlabel("true log2 fold change μ"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("SIMULATED: the posterior is a precision-weighted compromise")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Three replicates with SD 0.8 give the data a precision of $3/0.64 = 4.69$; the prior N(0, 0.5²)
has precision 4. The weight on the data is $4.69/8.69 = 0.540$, so the observed fold change 1.2 is shrunk to
$0.5396\times1.2 = 0.647$, with posterior SD $1/\sqrt{8.69} = 0.339$ — narrower than either the prior (0.5) or the likelihood
($0.8/\sqrt3 = 0.46$). Is shrinking honest? Over many genes, yes:

In [ ]:
# ── 9.2 Over 2,000 genes, shrunken fold changes are closer to the truth (SIMULATED, true LFC ~ N(0, 0.5²)) ──
rng_lfc = np.random.default_rng(SEED + 92)
true_lfc = rng_lfc.normal(0.0, 0.5, 2000)
xbar_all = rng_lfc.normal(true_lfc[:, None], sigma_rep, (2000, n_rep)).mean(axis=1)
shrunk = (prec_data * xbar_all + prec_prior * mu0) / (prec_data + prec_prior)
print(f"MSE raw x̄ = {np.mean((xbar_all - true_lfc) ** 2):.4f} (theory σ²/n = {sigma_rep ** 2 / n_rep:.4f});  "
      f"MSE shrunken = {np.mean((shrunk - true_lfc) ** 2):.4f} (theory posterior variance = {sd_post ** 2:.4f})")
top_raw, top_shr = np.argsort(-np.abs(xbar_all))[:20], np.argsort(-np.abs(shrunk))[:20]
print(f"the 20 genes with the largest |raw LFC|: mean |true LFC| = {np.abs(true_lfc[top_raw]).mean():.3f}, "
      f"mean |raw estimate| = {np.abs(xbar_all[top_raw]).mean():.3f}, mean |shrunken estimate| = {np.abs(shrunk[top_raw]).mean():.3f}")
print(f"the 20 genes with the largest |shrunken LFC| are the same genes: {set(top_raw) == set(top_shr)} "
      f"(every gene has the same precision, so shrinkage keeps the order)")

🔍 **What just happened?** Across 2,000 simulated genes the raw means miss the true fold changes with MSE 0.2130 (theory
0.2133), the shrunken estimates with MSE 0.1179 (theory 0.1151) — almost halving the error, as the posterior variance
predicts (up to simulation noise). Look at the 20 most extreme raw estimates: they average 1.922 in absolute value, while their true fold changes
average only 1.023 — the "winner's curse" of picking the largest noisy values. Their shrunken estimates (1.037) are far
closer to the truth. Here every gene has the same precision, so shrinkage keeps the order of the genes (the last line prints
`True`) and corrects only the *size* of the effects. In real data genes differ in noise (a gene with few reads is far noisier),
so shrinkage pulls the noisy genes down more and also changes the ranking — which is why DESeq2 reports shrunken fold changes.

In [ ]:
# ── 9.3 How fast the data take over: weight on the data and posterior SD as n grows (σ = 0.8, prior SD 0.5) ──
n_range = np.arange(1, 101)
weight = (n_range / sigma_rep ** 2) / (n_range / sigma_rep ** 2 + 1 / sigma0 ** 2)
post_sd = 1 / np.sqrt(n_range / sigma_rep ** 2 + 1 / sigma0 ** 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(n_range, weight, color="tab:blue"); axes[0].axhline(1, color="k", lw=0.6)
axes[0].set_xlabel("replicates n"); axes[0].set_ylabel(r"weight $w$ on $\bar x$")
axes[0].set_title(r"$\mu_{MAP} = w\,\bar x + (1 - w)\,\mu_0$")
axes[1].plot(n_range, sigma_rep / np.sqrt(n_range), color="tab:orange", label=r"SE of $\bar x$ (no prior)")
axes[1].plot(n_range, post_sd, color="tab:blue", label="posterior SD")
axes[1].set_xlabel("replicates n"); axes[1].set_ylabel("uncertainty about μ"); axes[1].legend()
axes[1].set_title("the prior matters most when n is small")
plt.tight_layout(); plt.show()
for n_ in [1, 3, 10, 100]:
    print(f"n = {n_:3d}: weight on the data {weight[n_ - 1]:.3f} | SE of x̄ {sigma_rep / np.sqrt(n_):.3f} | "
          f"posterior SD {post_sd[n_ - 1]:.3f}")

🔍 **What just happened?** With one replicate the data get weight 0.281 and the prior the rest; with 3 replicates 0.540, with 10
0.796 and with 100 0.975 — the MAP converges to the MLE $\bar x$. The posterior SD is always below the frequentist standard
error $\sigma/\sqrt n$ (the prior adds information), but the gap closes as $n$ grows (0.424 vs 0.800 at $n = 1$; 0.079 vs
0.080 at $n = 100$). A prior is most valuable exactly where biology is most often stuck: very few replicates.

### 🌉 Bridge — ridge regression is MAP with a Gaussian prior

Put a normal prior $\beta_j\sim\mathcal N(0, \tau^2)$ on every coefficient of a linear regression with Gaussian noise (Module 23's
least squares = MLE). The log-posterior is $-\frac{1}{2\sigma^2}\sum_i(y_i - \mathbf x_i^\top\boldsymbol\beta)^2 - \frac{1}{2\tau^2}\sum_j\beta_j^2$
+ constant, so the MAP minimises the residual sum of squares **plus** $\lambda\sum_j\beta_j^2$ with $\lambda = \sigma^2/\tau^2$ — ridge
regression, the standard fix when there are more genes than samples. A Laplace prior gives the lasso. See
[BiologyToML 07 · ridge, lasso and sparse regression](../../../../ml/tutorial/BiologyToML/notebooks/07_ridge_lasso_sparse_regression.ipynb),
[BiologyToML 05 · likelihood, MLE and MAP](../../../../ml/tutorial/BiologyToML/notebooks/05_likelihood_mle_map.ipynb), and Module 27.

✏️ **Your turn.** (a) By hand: one measurement $x = 10$ with $\sigma = 2$, prior N(6, 2²). MAP and posterior variance? (b) With
$n$ measurements of mean 10, how large must $n$ be for the MAP to be within 0.1 of 10?

### 📘 Lecture notes — MAP estimation and conjugate priors

**MAP.** $\hat\theta_{\text{MLE}} = \arg\max_\theta p(\mathcal D\mid\theta)$; $\hat\theta_{\text{MAP}} = \arg\max_\theta p(\theta\mid\mathcal D) =
\arg\max_\theta p(\mathcal D\mid\theta)p(\theta)$. Both are **point estimates**; MAP is **partially Bayesian** (it uses the prior but
not the whole posterior). With an unknown prior, choose one and solve $\frac{d}{d\theta}\ln[p(\mathcal D\mid\theta)p(\theta)] = 0$.

**Bernoulli + Beta($\alpha_{H_0}, \alpha_{T_0}$).** $\hat\theta_{\text{MAP}} = \dfrac{\alpha_H + \alpha_{H_0} - 1}{\alpha_H + \alpha_T + \alpha_{H_0} + \alpha_{T_0} - 2}
= \dfrac{k + (\alpha - 1)}{n + (\alpha - 1) + (\beta - 1)}$ (all counts with **+**). Flat prior ⇒ MAP = MLE.

**Discrete prior** ($x = 0,0,1,1,0$; $\theta\in\{0.2, 0.5, 0.7\}$; prior 0.1, 0.01, 0.89): likelihoods 0.02048, 0.03125, 0.01323;
numerators 0.002048, 0.0003125, 0.0117747 ($= 0.7^2 0.3^3\cdot 0.89$); posterior 0.145, 0.022 and 0.833; MAP 0.7 (MLE 0.5).

**Conjugate priors** — prior and posterior in the same family; prior parameters = **hyperparameters**; set them from experts or
use an uninformative prior (Beta(1, 1)); or estimate them across many genes (empirical Bayes). Beta–Bernoulli, Beta–binomial →
Beta; Gamma–Poisson → Gamma; Normal–Normal → Normal.

**Poisson + Gamma (shape $k$, scale $\theta$).** $\ln(P(\mathcal D\mid\lambda)P(\lambda)) = \ln\lambda(k - 1 + \sum x_i) - \lambda(n + 1/\theta) + \text{const}$;
$\hat\lambda_{\text{MAP}} = \dfrac{k - 1 + \sum_i x_i}{n + 1/\theta}$; posterior Gamma(shape $k + \sum x_i$, rate $n + 1/\theta$).

**Normal mean + Normal($\mu_0, \sigma_0^2$) prior, $\sigma^2$ known.** One observation: $\mu_{\text{MAP}} = \dfrac{x\sigma_0^2 + \mu_0\sigma^2}{\sigma_0^2 + \sigma^2}$;
$n$ observations: precision-weighted average of $\bar x$ (precision $n/\sigma^2$) and $\mu_0$ (precision $1/\sigma_0^2$); posterior
variance $1/(n/\sigma^2 + 1/\sigma_0^2)$ — a variance of $\mu$, not an estimate of $\sigma^2$. Shrinkage nearly halves the MSE of noisy fold
changes (simulation: 0.2130 → 0.1179).

**MLE vs MAP vs posterior mean.** Same for large $n$; with small $n$ a sensible prior reduces the error (Beta(3,7) at $n = 5$:
MSE 0.0072 vs 0.0419), a confident wrong prior increases it.

> 🎓 **Exam tip.** For any conjugate pair, be able to (1) write log(likelihood × prior), (2) drop constants, (3) differentiate
> and solve for the MAP, and (4) name the posterior family and its updated hyperparameters. Interpret the MAP as "data + imaginary
> data from the prior".

## 📌 Key takeaways

- The Bernoulli, binomial, geometric, Poisson, negative binomial, normal and exponential distributions belong to the
  **exponential family** $h(x)e^{\eta(\theta)\cdot T(x) - B(\theta)}$; the data enter the likelihood only through the **sufficient
  statistic** $\sum_i T(x_i)$, and one theory serves every member.
- **Bayes' theorem** for parameters: posterior ∝ likelihood × prior; with a **known prior** it is pure probability (screening test:
  0.199; drawer of coins: 0.641/0.308/0.051 after one tails; after 6H/4T MAP = A but MLE = B).
- **Frequentists** treat parameters as fixed and use the likelihood only; **Bayesians** treat them as random and need a prior —
  and in practice there is no universally accepted prior.
- **Pseudocounts** are imaginary data: $(c_H + c_{H_0})/(n + c_{H_0} + c_{T_0})$ (0 heads in 100: 0, 1/102, 1/3); they cure zero
  counts (motif matrices) and fade as data accumulate.
- **Beta prior + Bernoulli data → Beta posterior** Beta($k + \alpha$, $n - k + \beta$); posterior mean $(k+\alpha)/(n+\alpha+\beta)$, **MAP**
  $(k + \alpha - 1)/(n + \alpha + \beta - 2)$.
- **MAP** = $\arg\max p(\mathcal D\mid\theta)p(\theta)$, a point estimate — partially Bayesian. **Conjugate priors** (Beta, Gamma, Normal) make
  the posterior a matter of updating hyperparameters: Gamma–Poisson MAP $(k - 1 + \sum x)/(n + 1/\theta)$; Normal–Normal MAP = a
  **precision-weighted average** that **shrinks** noisy estimates.
- A prior is a bet: it helps with few data when sensible, hurts when confidently wrong, and is overwhelmed by large data.
  **Empirical Bayes** learns the prior from thousands of genes — the logic of DESeq2/edgeR/limma; **ridge = Gaussian-prior MAP**.

---
## ✅ Checkpoint

1. Write the Poisson PMF in exponential-family form and name $h$, $T$, $\eta$, $B$. What is the sufficient statistic for $n$ i.i.d. counts?
   <details><summary>Answer</summary>$e^{-\lambda}\lambda^x/x! = \frac{1}{x!}\exp(x\log\lambda - \lambda)$: $h = 1/x!$, $T = x$, $\eta = \log\lambda$, $B = \lambda$. For $n$ counts the likelihood depends on the data only through $\sum_i x_i$.</details>
2. In the normal exponential-family form, what is $B(\theta)$, and why is it not $-\frac{\mu^2}{2\sigma^2} - \log\sigma$?
   <details><summary>Answer</summary>$B = \frac{\mu^2}{2\sigma^2} + \log\sigma$. The exponent is $\eta\cdot T - B$, and the leftover terms in the exponent are $-\frac{\mu^2}{2\sigma^2} - \log\sigma$, which equal $-B$.</details>
3. Screening: sensitivity 0.95, false-positive rate 0.05, prevalence 1 %. $P(\text{ill}\mid +)$?
   <details><summary>Answer</summary>$0.0095/(0.0095 + 0.0495) = 0.161$.</details>
4. Drawer of 10 coins (5 A at 0.5, 3 B at 0.6, 2 C at 0.9), one flip gives **heads**. Posterior for A, B, C?
   <details><summary>Answer</summary>Numerators $0.25, 0.18, 0.18$ (sum 0.61) → $0.410, 0.295, 0.295$. A is still the MAP; compared with the prior (0.5, 0.3, 0.2), C gained most and A lost most, while B barely moved (0.300 → 0.295).</details>
5. 0 heads in 100 tosses. Give the MLE and the pseudocount estimates with $c_{H_0} = c_{T_0} = 1$ and $= 100$. Which prior Beta($\alpha, \beta$) makes each pseudocount estimate a posterior mean?
   <details><summary>Answer</summary>$0$; $1/102$; $1/3$. Posterior means under Beta(1, 1) and Beta(100, 100) respectively (posterior mean $=(k+\alpha)/(n+\alpha+\beta)$).</details>
6. Prior Beta(3, 3), data 4 heads and 3 tails. Posterior, MAP, posterior mean, MLE.
   <details><summary>Answer</summary>Beta(7, 6); MAP $6/11 = 0.545$; mean $7/13 = 0.538$; MLE $4/7 = 0.571$.</details>
7. Derive the Beta-prior MAP from $(1-p)(\alpha_H + \alpha_{H_0} - 1) = p(\alpha_T + \alpha_{T_0} - 1)$. What sign does $\alpha_{T_0}$ have in the denominator?
   <details><summary>Answer</summary>$\alpha_H + \alpha_{H_0} - 1 - p(\alpha_H + \alpha_{H_0} - 1) = p(\alpha_T + \alpha_{T_0} - 1) \Rightarrow \alpha_H + \alpha_{H_0} - 1 = p(\alpha_H + \alpha_{H_0} + \alpha_T + \alpha_{T_0} - 2)$, so $\hat p = (\alpha_H + \alpha_{H_0} - 1)/(\alpha_H + \alpha_T + \alpha_{H_0} + \alpha_{T_0} - 2)$ — plus.</details>
8. Discrete prior: $x = (1, 1, 0)$, $\theta\in\{0.3, 0.6\}$, prior 0.7, 0.3. MLE and MAP?
   <details><summary>Answer</summary>Likelihoods $0.3^2\cdot0.7 = 0.063$ and $0.6^2\cdot0.4 = 0.144$ → MLE 0.6. Numerators $0.0441$ and $0.0432$ → posterior $0.505, 0.495$ → MAP 0.3 (narrowly).</details>
9. Counts 0, 1, 0 in three cells; prior Gamma(shape 3, rate 1). Posterior, MAP, posterior mean.
   <details><summary>Answer</summary>Gamma(shape $3 + 1 = 4$, rate $1 + 3 = 4$); MAP $(4 - 1)/4 = 0.75$; mean $4/4 = 1$ (MLE $1/3$).</details>
10. One measurement $x = 10$ ($\sigma = 2$), prior N(6, 2²). MAP and posterior variance. With $n = 4$ measurements of mean 10?
    <details><summary>Answer</summary>Equal precisions $1/4$ each: MAP $= 8$, posterior variance $1/(1/4 + 1/4) = 2$. With $n = 4$: data precision $4/4 = 1$, prior $1/4$ → MAP $= (1\cdot10 + 0.25\cdot6)/1.25 = 9.2$, variance $0.8$.</details>
11. Why is MAP called "partially Bayesian"? Give a case where the MAP is a poor summary of the posterior.
    <details><summary>Answer</summary>It uses the prior but reduces the posterior to its peak, discarding its spread. Example: 0 variants in 100 chromosomes with a flat prior — MAP 0, while the posterior mean is 0.0098 and the 95 % upper bound 0.029.</details>
12. In one paragraph: why does ranking genes by raw fold changes from 3 replicates exaggerate the top hits, and how does a prior help?
    <details><summary>Answer</summary>The most extreme raw estimates are disproportionately those with large positive noise (winner's curse), so their true effects are smaller. A prior centred near 0 shrinks each estimate by a precision-weighted amount — noisy estimates more — which reduces the mean squared error and makes rankings more reliable.</details>

## 🧠 Questions & discussion

1. A frequentist says "a prior is subjective". A Bayesian replies "so is the choice of model, test and significance level". Where
   in your own field's analyses are such choices made, and how would you report them?
2. How would you elicit a prior for the frequency of a resistance gene in a new hospital? Discuss surveillance data from other
   hospitals, expert opinion, and the risk of a confidently wrong prior (Section 7.4).
3. Empirical Bayes uses the same data twice — to learn the prior and to compute each gene's posterior. Why is this usually fine
   with 20,000 genes, and when might it fail?
4. Pseudocounts of 1 per base are a convention in motif building. Argue for a pseudocount proportional to the background base
   frequency instead. What happens in a very GC-rich genome?
5. Credible intervals answer the question scientists usually want answered ("where is the parameter?"). Why do so many
   journals still require confidence intervals and p-values?

## 🚀 Bonus — going further

- **MAP is not invariant.** Compute the MAP of the log-odds $\phi = \log\frac{p}{1-p}$ under the prior induced by Beta(3, 7) (change
  of variables: multiply by $|dp/d\phi| = p(1-p)$) and transform back. Is it the MAP of $p$? (The MLE, in contrast, is invariant.)
- **Posterior predictive.** Under a Beta posterior, the number of resistant isolates among the next $m$ follows a Beta-binomial
  (`scipy.stats.betabinom`). Compare it with Binomial($m$, MLE) for $m = 20$ after 3 of 20.
- **Jeffreys prior.** For the Bernoulli it is Beta(½, ½). Compare the coverage of its 95 % credible interval with the Wald interval of
  Module 23 for $n = 20$, $p = 0.05$ by simulation.
- **Dirichlet–multinomial.** Generalise Beta–binomial to $K$ categories (the four bases of a motif position): prior Dirichlet($\alpha_1..\alpha_K$),
  posterior Dirichlet($\alpha_k + n_k$). Recompute the PWM of Section 4 as posterior means.
- **When there is no conjugate prior,** sample the posterior: write a 20-line Metropolis sampler for the ZIP model of Module 23
  ([BiologyToML 21](../../../../ml/tutorial/BiologyToML/notebooks/21_bayesian_learning_mcmc.ipynb)).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module:            24 · Bayesian estimation and maximum a posteriori
Input:             typed-in worked examples (screening test, drawer of 10 coins, discrete prior x = 0,0,1,1,0, ten AP-1-like
                   binding sites); SIMULATED coin flips, isolates, rare-variant counts, expression counts, fold changes
Output:            exponential-family forms, Bayes tables, pseudocount estimates, Beta/Gamma/Normal posteriors, MAP estimates,
                   credible intervals, MSE comparisons, empirical-Bayes and shrunken estimates
Method:            posterior ∝ likelihood × prior; conjugate updating of hyperparameters; d/dθ log(likelihood × prior) = 0
Key equation:      θ̂_MAP = argmax p(D|θ)p(θ);  Beta: (k + α − 1)/(n + α + β − 2);  Gamma: (k − 1 + Σx)/(n + 1/θ);
                   Normal: (n x̄/σ² + μ0/σ0²)/(n/σ² + 1/σ0²)
Key parameters:    hyperparameters (prior strength and centre); sample size n
What changed when I modified …  (e.g. Beta(1,1) → Beta(40,40); pseudocount 0 → 1; n = 5 → 100; 3 → 30 cells per gene)
One failure mode:  a confidently wrong prior dominates small data (Beta(14,6) when the truth is 0.3)
Connection to 02-680: exponential family, Bayes with a known prior, frequentist vs Bayesian, pseudocounts, Beta, conjugacy,
                   MAP (known prior, unknown prior, discrete prior), Gamma–Poisson and Normal–Normal MAP
```

## 📚 References

- Wasserman L. (2004) *All of Statistics: A Concise Course in Statistical Inference*. Springer — Ch. 11 (Bayesian inference: the
  Bayesian method, conjugate priors, flat and Jeffreys priors, frequentist vs Bayesian).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — §§7.1–7.4 (statistical inference, prior and
  posterior distributions, conjugate prior distributions, Bayes estimators).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press — §6.6 (conjugacy and
  the exponential family), §8.3.2 (maximum a posteriori estimation).
- Betti M. *Mathematics and Statistics for the Quantitative Sciences*. CRC Press — the chapters on Bayesian estimation.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.
- Robinson M. D., McCarthy D. J., Smyth G. K. (2010) edgeR: a Bioconductor package for differential expression analysis of digital
  gene expression data. *Bioinformatics* 26:139–140.
- Love M. I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. *Genome
  Biology* 15:550.
- Smyth G. K. (2004) Linear models and empirical Bayes methods for assessing differential expression in microarray experiments.
  *Statistical Applications in Genetics and Molecular Biology* 3: Article 3.

## ➡️ Next

**[Module 25 · 🚦 Hypothesis testing: errors, power, z- and t-tests](25_hypothesis_testing.ipynb)** — Modules 22–24 estimated parameters and their uncertainty. Module 25 asks yes-or-no questions about them:
*does* the drug change cholesterol? *Is* the coin unfair? Hypothesis testing uses the sampling distributions of Module 22 (the
z statistic is the standardised mean; the t statistic uses the sample SD), the likelihood ideas of Module 23, and keeps the
frequentist reading of probability that Section 3 contrasted with the Bayesian one — with Type I and Type II errors, power, and
p-values that, unlike posterior probabilities, are *not* the probability that the hypothesis is true.